# Umbral, métricas y artefactos

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Tipos de ataque

`NORMAL_CAT` y `ATTACK_TYPES` guardan los valores de `attack_cat`: el tráfico normal y los 9 tipos de ataque, en el orden del contrato. Las métricas del tipo de ataque los usan para listar siempre los 9 tipos, en el mismo orden.

Comprobamos que coinciden exactamente con los valores de `attack_cat` del archivo de entrenamiento y del de prueba (sin faltar ni sobrar ninguno), y que en los dos archivos `label` = 1 equivale a que `attack_cat` sea uno de los 9 tipos de ataque. Aquí la prueba solo se abre para leer sus categorías.

In [2]:
from threat_detector.dataset import ATTACK_CAT, ATTACK_TYPES, NORMAL_CAT, TARGET, load_test, load_train

expected = {NORMAL_CAT, *ATTACK_TYPES}
print(f"Tipos de ataque: {len(ATTACK_TYPES)}; ¿sin repetidos?: {len(set(ATTACK_TYPES)) == len(ATTACK_TYPES)}")

for name, flows in (("Entrenamiento", load_train()), ("Prueba", load_test())):
    found = set(flows[ATTACK_CAT].unique())
    label_matches = (flows[TARGET] == 1).equals(flows[ATTACK_CAT].isin(ATTACK_TYPES))
    print(f"{name}: ¿mismos valores de attack_cat?: {found == expected} "
          f"(faltan: {sorted(expected - found)}, sobran: {sorted(found - expected)}); "
          f"¿label = 1 si y solo si es un tipo de ataque?: {label_matches}")

Tipos de ataque: 9; ¿sin repetidos?: True
Entrenamiento: ¿mismos valores de attack_cat?: True (faltan: [], sobran: []); ¿label = 1 si y solo si es un tipo de ataque?: True
Prueba: ¿mismos valores de attack_cat?: True (faltan: [], sobran: []); ¿label = 1 si y solo si es un tipo de ataque?: True


## 2. Alerta y severidad

`is_alert` es la única regla de alerta del proyecto: un flujo es alerta si su probabilidad de ataque es **mayor o igual** que el umbral del modelo. `severity` la usa como primer paso y después compara con los cortes de `config.py` (`SEVERITY_CRITICAL`, `SEVERITY_HIGH`, `SEVERITY_MEDIUM`). Las dos están en `artifacts.py`, para que la app y `ml/` usen exactamente la misma regla.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos probabilidades y umbrales escritos a mano, elegidos justo en los límites. Las funciones se aplicarán a las probabilidades reales de cada modelo cuando esté entrenado.

**Resultado esperado:**
- Con umbral 0,31: una probabilidad justo en el umbral es `"low"`, y justo en cada corte sube de nivel (0,60 → `"medium"`, 0,75 → `"high"`, 0,90 → `"critical"`); un poco por debajo de cada uno se queda en el nivel anterior.
- Con umbral 0,70 (mayor que 0,60) ninguna probabilidad da `"low"`: el nivel queda vacío, y es correcto.
- `is_alert` es verdadero si y solo si la severidad no es `"normal"`; con muchos flujos devuelve un valor por flujo, y con uno solo devuelve un `bool` de Python que se puede escribir en JSON.

In [3]:
import json

import numpy as np
import pandas as pd

from threat_detector.artifacts import is_alert, severity
from threat_detector.config import SEVERITY_CRITICAL, SEVERITY_HIGH, SEVERITY_MEDIUM

# Probabilidades inventadas con la severidad que debe salir, para dos umbrales inventados.
cases = {
    0.31: [(0.30, "normal"), (0.31, "low"), (0.5999, "low"), (SEVERITY_MEDIUM, "medium"),
           (0.7499, "medium"), (SEVERITY_HIGH, "high"), (0.8999, "high"),
           (SEVERITY_CRITICAL, "critical"), (1.0, "critical")],
    0.70: [(0.65, "normal"), (0.70, "medium"), (0.7499, "medium"), (0.80, "high"), (0.95, "critical")],
}
rows = [
    {"umbral": threshold, "probabilidad": p, "esperada": expected, "obtenida": severity(p, threshold),
     "is_alert": is_alert(p, threshold)}
    for threshold, pairs in cases.items()
    for p, expected in pairs
]
results = pd.DataFrame(rows)
results["¿coincide?"] = results["esperada"] == results["obtenida"]

print(f"¿Todas las severidades coinciden?: {results['¿coincide?'].all()}")
print(f"¿Algún 'low' con umbral 0,70?: {'low' in results.loc[results['umbral'] == 0.70, 'obtenida'].tolist()}")
print(f"¿is_alert si y solo si la severidad no es 'normal'?: {(results['is_alert'] == (results['obtenida'] != 'normal')).all()}")

many = is_alert(np.array([0.10, 0.31, 0.95]), 0.31)
one = is_alert(np.float64(0.5), 0.31)
print(f"is_alert con 3 flujos: {many}; con un flujo: {one!r} (tipo {type(one).__name__})")
print(f"Un flujo en JSON: {json.dumps({'is_alert': one, 'severity': severity(0.5, 0.31)})}")

results

¿Todas las severidades coinciden?: True
¿Algún 'low' con umbral 0,70?: False
¿is_alert si y solo si la severidad no es 'normal'?: True
is_alert con 3 flujos: [False  True  True]; con un flujo: True (tipo bool)
Un flujo en JSON: {"is_alert": true, "severity": "low"}


,umbral,probabilidad,esperada,obtenida,is_alert,¿coincide?
0,0.31,0.3000,normal,normal,False,True
1,0.31,0.3100,low,low,True,True
2,0.31,0.5999,low,low,True,True
3,0.31,0.6000,medium,medium,True,True
4,0.31,0.7499,medium,medium,True,True
5,0.31,0.7500,high,high,True,True
6,0.31,0.8999,high,high,True,True
7,0.31,0.9000,critical,critical,True,True
8,0.31,1.0000,critical,critical,True,True
9,0.70,0.6500,normal,normal,False,True


## 3. Tasas de detección y de falsas alarmas

`detection_rate` es la parte de los ataques reales (`label` = 1) que dio alerta, y `false_alarm_rate` la parte de los flujos normales reales (`label` = 0) que dio alerta. Están en `artifacts.py` porque las usan la app (en los indicadores de un CSV con etiquetas) y `ml/evaluation.py` (en validación y prueba). Si no hay flujos de la clase que se divide, devuelven `None` (`null` en el JSON), como dice el contrato para el CSV del usuario.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos etiquetas y alertas escritas a mano.

**Resultado esperado:**
- 4 ataques reales, 3 con alerta, y 5 normales, 1 con alerta: detección = 3/4 = 0,75 y falsas alarmas = 1/5 = 0,2.
- Un archivo con solo ataques: falsas alarmas = `None`. Un archivo con solo normales: detección = `None`.
- Los resultados son números de Python y se pueden escribir en JSON.

In [4]:
from threat_detector.artifacts import detection_rate, false_alarm_rate

# Etiquetas reales y alertas inventadas, con el resultado esperado.
rate_cases = [
    ("4 ataques (3 alertas) y 5 normales (1 alerta)", [1, 1, 1, 1, 0, 0, 0, 0, 0],
     [True, True, True, False, True, False, False, False, False], 0.75, 0.2),
    ("Solo ataques (1 de 2 con alerta)", [1, 1], [True, False], 0.5, None),
    ("Solo normales (ninguno con alerta)", [0, 0, 0], [False, False, False], None, 0.0),
]
for name, y_true, alerts, expected_detection, expected_false_alarm in rate_cases:
    detection = detection_rate(y_true, alerts)
    false_alarm = false_alarm_rate(y_true, alerts)
    ok = detection == expected_detection and false_alarm == expected_false_alarm
    print(f"{name}: detección = {detection}, falsas alarmas = {false_alarm}; ¿lo esperado?: {ok}; "
          f"JSON: {json.dumps({'detection_rate': detection, 'false_alarm_rate': false_alarm})}")

4 ataques (3 alertas) y 5 normales (1 alerta): detección = 0.75, falsas alarmas = 0.2; ¿lo esperado?: True; JSON: {"detection_rate": 0.75, "false_alarm_rate": 0.2}
Solo ataques (1 de 2 con alerta): detección = 0.5, falsas alarmas = None; ¿lo esperado?: True; JSON: {"detection_rate": 0.5, "false_alarm_rate": null}
Solo normales (ninguno con alerta): detección = None, falsas alarmas = 0.0; ¿lo esperado?: True; JSON: {"detection_rate": null, "false_alarm_rate": 0.0}


## 4. Umbral

`compute_threshold` calcula el umbral de alerta de un modelo con las predicciones de validación: el umbral **más alto** que deja al menos `TARGET_RECALL` (95 %) de los ataques reales como alerta. Ordena las probabilidades de los ataques reales de mayor a menor y toma la del puesto *k*, donde *k* es el número de ataques que deben quedar como alerta (95 % redondeado hacia arriba). Los flujos normales no cambian el umbral: las falsas alarmas son consecuencia, no objetivo.

> **Datos inventados.** Como aún no hay modelos entrenados, en los casos 1, 3, 4 y 5 usamos probabilidades escritas a mano (redondeadas a 2 decimales, para que el punto flotante no cambie los valores esperados). La función se aplicará a las probabilidades de validación de cada modelo cuando esté entrenado.

**Resultado esperado:**
1. **20 ataques** con probabilidades 0,05; 0,10; …; 1,00 y 10 normales: hacen falta 19 ataques, así que el umbral es **0,10** (la 19.ª más alta). Con 0,10 la detección es 19/20 = 0,95; con 0,1001 baja a 18/20 = 0,90; el ataque con probabilidad exactamente 0,10 es alerta.
2. **El cálculo es exacto:** el número de ataques que deben quedar como alerta se calcula como `ceil(TARGET_RECALL × ataques)`. Como `TARGET_RECALL` es un número decimal guardado en punto flotante, comprobamos para todos los tamaños de 1 a 50.000 ataques (la validación tiene unos 24.000) que ese cálculo da lo mismo que el cálculo exacto con fracciones. Debe coincidir en todos. Esta prueba confirma que el cálculo simple es exacto **para el valor configurado** de `TARGET_RECALL`; si `TARGET_RECALL` cambia en `config.py`, hay que volver a correrla.
3. **Empate en el umbral:** si dos ataques tienen 0,10, el umbral sigue siendo 0,10 y la detección sube a 20/20 = 1,0. Lo calculamos con dos grupos distintos de normales (probabilidades muy bajas y muy altas): el umbral es el mismo, porque los normales no lo cambian.
4. **7 ataques:** 0,95 × 7 = 6,65, así que hacen falta los 7: el umbral es la probabilidad más baja de los ataques (0,40) y la detección es 1,0. También con dos grupos distintos de normales, y el umbral no cambia.
5. **Errores:** etiquetas de una sola clase (la función exige flujos normales y ataques reales, igual que `threshold_rule`, que se calcula con los mismos datos de validación) y distinta cantidad de etiquetas y probabilidades. Los dos dan un mensaje en español.

In [5]:
import math
from fractions import Fraction

from threat_detector.artifacts import detection_rate
from threat_detector.config import TARGET_RECALL
from threat_detector.ml.evaluation import compute_threshold

# Caso 1: 20 ataques con probabilidades 0,05 a 1,00 (redondeadas a 2 decimales) y 10 normales.
attack_probs = np.round(np.arange(1, 21) * 0.05, 2)
normal_probs = np.round(np.arange(10) * 0.03, 2)
y_true = np.array([1] * 20 + [0] * 10)
probabilities = np.concatenate([attack_probs, normal_probs])

threshold = compute_threshold(y_true, probabilities)
print(f"Caso 1: umbral = {threshold} (esperado 0.1)")
print(f"    detección con el umbral: {detection_rate(y_true, is_alert(probabilities, threshold))} (esperada 0.95)")
print(f"    detección con 0.1001: {detection_rate(y_true, is_alert(probabilities, 0.1001))} (esperada 0.9)")
print(f"    ¿el ataque con probabilidad 0.10 es alerta?: {is_alert(0.10, threshold)}")

# Caso 2: el cálculo con punto flotante da lo mismo que el cálculo exacto con fracciones.
# Fraction(str(...)) toma el valor decimal escrito en config.py (por ejemplo 0.95 = 19/20).
exact_recall = Fraction(str(TARGET_RECALL))
mismatches = [n for n in range(1, 50_001) if math.ceil(TARGET_RECALL * n) != math.ceil(exact_recall * n)]
print(f"\nCaso 2: TARGET_RECALL = {TARGET_RECALL} = {exact_recall}; "
      f"tamaños de 1 a 50.000 ataques donde el cálculo simple no es exacto: {len(mismatches)} (esperado 0)")

# Casos 3 y 4: dos grupos de normales inventados, con probabilidades muy bajas y muy altas.
normal_groups = {"normales bajos": [0.01, 0.02, 0.03], "normales altos": [0.97, 0.98, 0.99]}

# Caso 3: dos ataques empatados en 0,10.
tied = [0.10, 0.10] + [round(0.05 * i, 2) for i in range(3, 21)]
print()
for group, normals in normal_groups.items():
    labels = [1] * 20 + [0] * len(normals)
    probs = tied + normals
    tied_threshold = compute_threshold(labels, probs)
    print(f"Caso 3 ({group}): umbral = {tied_threshold} (esperado 0.1); "
          f"detección = {detection_rate(labels, is_alert(probs, tied_threshold))} (esperada 1.0)")

# Caso 4: 7 ataques, hacen falta los 7.
seven = [0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
print()
for group, normals in normal_groups.items():
    labels = [1] * 7 + [0] * len(normals)
    probs = seven + normals
    seven_threshold = compute_threshold(labels, probs)
    print(f"Caso 4 ({group}): umbral = {seven_threshold} (esperado 0.4); "
          f"detección = {detection_rate(labels, is_alert(probs, seven_threshold))} (esperada 1.0)")

# Caso 5: errores.
print()
for name, bad_labels, bad_probs in (
    ("Solo normales", [0, 0, 0], [0.1, 0.2, 0.3]),
    ("Distinta cantidad", [1, 0, 1], [0.1, 0.2]),
):
    try:
        compute_threshold(bad_labels, bad_probs)
    except ValueError as error:
        print(f"Caso 5, {name}: {error}")

Caso 1: umbral = 0.1 (esperado 0.1)
    detección con el umbral: 0.95 (esperada 0.95)
    detección con 0.1001: 0.9 (esperada 0.9)
    ¿el ataque con probabilidad 0.10 es alerta?: True

Caso 2: TARGET_RECALL = 0.95 = 19/20; tamaños de 1 a 50.000 ataques donde el cálculo simple no es exacto: 0 (esperado 0)

Caso 3 (normales bajos): umbral = 0.1 (esperado 0.1); detección = 1.0 (esperada 1.0)
Caso 3 (normales altos): umbral = 0.1 (esperado 0.1); detección = 1.0 (esperada 1.0)

Caso 4 (normales bajos): umbral = 0.4 (esperado 0.4); detección = 1.0 (esperada 1.0)
Caso 4 (normales altos): umbral = 0.4 (esperado 0.4); detección = 1.0 (esperada 1.0)

Caso 5, Solo normales: Estas métricas necesitan flujos normales y ataques reales; los datos recibidos solo tienen una clase.
Caso 5, Distinta cantidad: Las etiquetas traen 3 flujos y las probabilidades 2; deben traer los mismos flujos.


## 5. Regla del umbral

`threshold_rule` arma el objeto `threshold_rule` de `model.json` (contrato, sección 11): qué detección mínima se exigió, que el umbral se calculó con la validación, cuántos flujos de validación se usaron y qué detección y falsas alarmas logró ese umbral. Las dos tasas salen de `detection_rate` y `false_alarm_rate` de `artifacts.py`, sin calcularlas otra vez.

Antes de calcular, comprueba que las etiquetas y las probabilidades traen los mismos flujos y que hay flujos normales y ataques reales; si no, lanza un error en español.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos los mismos datos escritos a mano del caso 1 de la sección 4: 20 ataques y 10 normales, con umbral 0,10.

**Resultado esperado:**
- `target_recall` = `TARGET_RECALL` (0,95) y `computed_on` = `"validation"`.
- `validation_flows` = 30.
- `validation_detection_rate` = 19/20 = 0,95.
- `validation_false_alarm_rate` = 0,6: los normales tienen probabilidades 0; 0,03; …; 0,27, y 6 de los 10 son ≥ 0,10.
- Las claves son exactamente las del contrato y el resultado se puede escribir en JSON.
- Con etiquetas de una sola clase, o con distinta cantidad de etiquetas y probabilidades, un error en español.

In [6]:
from threat_detector.ml.evaluation import threshold_rule

rule = threshold_rule(y_true, probabilities, threshold)
expected_rule = {
    "target_recall": TARGET_RECALL,
    "computed_on": "validation",
    "validation_flows": 30,
    "validation_detection_rate": 0.95,
    "validation_false_alarm_rate": 0.6,
}
print(f"threshold_rule: {json.dumps(rule)}")
print(f"¿Es lo esperado?: {rule == expected_rule}")
print(f"¿Claves en el orden del contrato?: {list(rule) == list(expected_rule)}")

for name, labels, probs in (
    ("Solo ataques", [1, 1, 1], [0.2, 0.5, 0.9]),
    ("Distinta cantidad", [1, 0, 1], [0.2, 0.5]),
):
    try:
        threshold_rule(labels, probs, 0.3)
    except ValueError as error:
        print(f"{name}: {error}")

threshold_rule: {"target_recall": 0.95, "computed_on": "validation", "validation_flows": 30, "validation_detection_rate": 0.95, "validation_false_alarm_rate": 0.6}
¿Es lo esperado?: True
¿Claves en el orden del contrato?: True
Solo ataques: Estas métricas necesitan flujos normales y ataques reales; los datos recibidos solo tienen una clase.
Distinta cantidad: Las etiquetas traen 3 flujos y las probabilidades 2; deben traer los mismos flujos.


## 6. Métricas de detección

`detection_metrics` arma el objeto `detection` de las métricas de prueba (contrato, sección 7). Compara las alertas (`is_alert` con el umbral del modelo) con las etiquetas reales: un ataque real con alerta es un verdadero positivo (`tp`), un normal con alerta un falso positivo (`fp`), un normal sin alerta un verdadero negativo (`tn`) y un ataque sin alerta un falso negativo (`fn`). La detección y las falsas alarmas salen de `artifacts.py`; `pr_auc` y `roc_auc` no usan el umbral, sino las probabilidades.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos probabilidades escritas a mano, elegidas para poder calcular cada métrica a mano. La función se aplicará a las probabilidades reales de la prueba de cada modelo cuando esté entrenado.

**Resultado esperado:**
1. **Caso general:** 5 ataques con probabilidades 0,90; 0,80; 0,70; 0,60; 0,20 y 5 normales con 0,65; 0,30; 0,20; 0,10; 0,05, con umbral 0,50.
   - `tp` = 4 (0,90 a 0,60), `fn` = 1 (0,20), `fp` = 1 (0,65), `tn` = 4.
   - Detección = 4/5 = 0,8; falsas alarmas = 1/5 = 0,2; precisión = 4/5 = 0,8; `f1` = 2·4 / (2·4 + 1 + 1) = 0,8; `accuracy` = 8/10 = 0,8.
   - `roc_auc` = **0,86**: de los 25 pares ataque–normal, el ataque tiene la probabilidad más alta en 21 y hay un empate (0,20 y 0,20) que cuenta medio: 21,5 / 25.
   - `pr_auc` = **0,885**: recorriendo las probabilidades de mayor a menor, cada vez que la detección sube 0,2 se suma 0,2 × la precisión en ese punto: 0,2·1 + 0,2·1 + 0,2·1 + 0,2·0,8 + 0,2·0,625.
2. **Separación perfecta** (todos los ataques por encima de todos los normales): `pr_auc` = `roc_auc` = 1,0.
3. **Sin ninguna alerta** (umbral 0,99): `precision` = `None` (`null` en el JSON), `f1` = 0, detección = 0 y falsas alarmas = 0.
4. **Errores:** etiquetas de una sola clase, o distinta cantidad de etiquetas y probabilidades.
5. Las claves están en el orden del contrato y el resultado se puede escribir en JSON.

In [7]:
from threat_detector.ml.evaluation import detection_metrics

# Caso 1: probabilidades inventadas con resultado calculado a mano.
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
probs = [0.90, 0.80, 0.70, 0.60, 0.20, 0.65, 0.30, 0.20, 0.10, 0.05]
detection = detection_metrics(labels, probs, 0.50)
expected_detection = {
    "detection_rate": 0.8, "false_alarm_rate": 0.2, "precision": 0.8, "f1": 0.8,
    "pr_auc": 0.885, "roc_auc": 0.86, "accuracy": 0.8,
    "confusion": {"tp": 4, "fp": 1, "tn": 4, "fn": 1},
}
print(f"Caso 1: {json.dumps(detection)}")
close = all(
    detection[key] == value if key == "confusion" else math.isclose(detection[key], value)
    for key, value in expected_detection.items()
)
print(f"    ¿Es lo esperado?: {close}; ¿claves en el orden del contrato?: {list(detection) == list(expected_detection)}")

# Caso 2: separación perfecta.
perfect = detection_metrics([1, 1, 0, 0], [0.9, 0.8, 0.3, 0.1], 0.50)
print(f"\nCaso 2: pr_auc = {perfect['pr_auc']}, roc_auc = {perfect['roc_auc']} (esperados 1.0)")

# Caso 3: ninguna alerta.
no_alerts = detection_metrics(labels, probs, 0.99)
print(f"\nCaso 3: precision = {no_alerts['precision']}, f1 = {no_alerts['f1']}, "
      f"detección = {no_alerts['detection_rate']}, falsas alarmas = {no_alerts['false_alarm_rate']}")
print(f"    En JSON: {json.dumps({'precision': no_alerts['precision'], 'f1': no_alerts['f1']})}")

# Caso 4: errores.
for name, bad_labels, bad_probs in (
    ("Solo normales", [0, 0, 0], [0.2, 0.5, 0.9]),
    ("Distinta cantidad", [1, 0, 1, 0], [0.2, 0.5, 0.9]),
):
    try:
        detection_metrics(bad_labels, bad_probs, 0.50)
    except ValueError as error:
        print(f"\nCaso 4, {name}: {error}")

Caso 1: {"detection_rate": 0.8, "false_alarm_rate": 0.2, "precision": 0.8, "f1": 0.8, "pr_auc": 0.885, "roc_auc": 0.86, "accuracy": 0.8, "confusion": {"tp": 4, "fp": 1, "tn": 4, "fn": 1}}
    ¿Es lo esperado?: True; ¿claves en el orden del contrato?: True

Caso 2: pr_auc = 1.0, roc_auc = 1.0 (esperados 1.0)

Caso 3: precision = None, f1 = 0.0, detección = 0.0, falsas alarmas = 0.0
    En JSON: {"precision": null, "f1": 0.0}

Caso 4, Solo normales: Estas métricas necesitan flujos normales y ataques reales; los datos recibidos solo tienen una clase.

Caso 4, Distinta cantidad: Las etiquetas traen 4 flujos y las probabilidades 3; deben traer los mismos flujos.


## 7. Métricas del tipo de ataque

`attack_type_metrics` arma el objeto `attack_type` de las métricas de prueba (contrato, sección 7). Solo evalúa los **ataques reales que dieron alerta**, porque el tipo probable solo existe en las alertas; por eso `evaluated_flows` es igual a `tp` de `detection_metrics`. Compara el tipo real con el predicho entre los 9 tipos de `ATTACK_TYPES` y devuelve, para cada tipo, precisión, recall, `f1` y flujos reales (`support`), el promedio `macro_f1` y la matriz de confusión de 9 × 9 (fila = tipo real, columna = tipo predicho).

Cuando una métrica no se puede calcular es `None` (`null` en el JSON): la precisión de un tipo que nunca se predijo, el recall de un tipo sin flujos reales, y el `f1` de un tipo que no aparece ni en lo real ni en lo predicho. `macro_f1` promedia solo los `f1` que se pueden calcular.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos tipos reales, tipos predichos y alertas escritos a mano, elegidos para poder calcular cada métrica a mano. La función se aplicará a las predicciones reales de la prueba de cada modelo cuando esté entrenado.

| Flujos | Tipo real | Tipo predicho | ¿Alerta? |
|---|---|---|---|
| 3 | Generic | Generic | Sí |
| 1 | Generic | Exploits | Sí |
| 2 | Exploits | Exploits | Sí |
| 1 | Exploits | Generic | Sí |
| 1 | DoS | Exploits | Sí |
| 1 | Fuzzers | — | No (queda fuera: no es alerta) |
| 1 | Normal | Generic | Sí (queda fuera: no es un ataque real) |

**Resultado esperado:**
- `evaluated_flows` = 8, igual a `tp` de `detection_metrics` con las mismas alertas.
- Generic: precisión = 3/4 = 0,75, recall = 3/4 = 0,75, `f1` = 0,75, `support` = 4.
- Exploits: precisión = 2/4 = 0,5, recall = 2/3, `f1` = 2·2 / (3 + 4) = 4/7 ≈ 0,571, `support` = 3.
- DoS (nunca predicho): precisión `None`, recall 0, `f1` 0, `support` = 1.
- Los otros 6 tipos (incluido Fuzzers, porque su único flujo no es alerta, y Worms, que no aparece): todo `None` y `support` = 0.
- `macro_f1` = (0,75 + 4/7 + 0) / 3 ≈ 0,440.
- Matriz de 9 × 9: [Generic][Generic] = 3, [Generic][Exploits] = 1, [Exploits][Exploits] = 2, [Exploits][Generic] = 1, [DoS][Exploits] = 1 y el resto 0.
- Sin ningún flujo evaluado: `evaluated_flows` = 0 y `macro_f1` = `None`.
- Errores en español: un ataque real evaluado con un tipo que no es de los 9, una alerta con tipo probable `"Normal"` y distinta cantidad de flujos.

In [8]:
from threat_detector.dataset import ATTACK_TYPES
from threat_detector.ml.evaluation import attack_type_metrics

rows = (
    [("Generic", "Generic", True)] * 3 + [("Generic", "Exploits", True)]
    + [("Exploits", "Exploits", True)] * 2 + [("Exploits", "Generic", True)]
    + [("DoS", "Exploits", True), ("Fuzzers", None, False), ("Normal", "Generic", True)]
)
true_cats, predicted_cats, alerts = (list(column) for column in zip(*rows))
attack_type = attack_type_metrics(true_cats, predicted_cats, alerts)

# El mismo caso visto como detección: probabilidad 0,9 si es alerta y 0,1 si no, con umbral 0,5.
type_labels = [int(cat != NORMAL_CAT) for cat in true_cats]
type_detection = detection_metrics(type_labels, [0.9 if a else 0.1 for a in alerts], 0.5)
print(f"evaluated_flows = {attack_type['evaluated_flows']}; tp de detection_metrics = {type_detection['confusion']['tp']} (esperados 8)")

by_type = {c["attack_cat"]: c for c in attack_type["per_class"]}
expected_types = {
    "Generic": {"precision": 0.75, "recall": 0.75, "f1": 0.75, "support": 4},
    "Exploits": {"precision": 0.5, "recall": 2 / 3, "f1": 4 / 7, "support": 3},
    "DoS": {"precision": None, "recall": 0.0, "f1": 0.0, "support": 1},
}
empty = {"precision": None, "recall": None, "f1": None, "support": 0}
types_ok = all(
    {k: v for k, v in by_type[cat].items() if k != "attack_cat"} == expected_types.get(cat, empty)
    for cat in ATTACK_TYPES
)
print(f"¿Métricas de los 9 tipos como se esperaba?: {types_ok}; "
      f"¿los 9 en orden?: {[c['attack_cat'] for c in attack_type['per_class']] == ATTACK_TYPES}")
print(f"macro_f1 = {attack_type['macro_f1']:.4f} (esperado {(0.75 + 4 / 7 + 0) / 3:.4f})")

matrix = pd.DataFrame(attack_type["confusion"]["matrix"], index=ATTACK_TYPES, columns=ATTACK_TYPES)
print(f"Matriz: {matrix.shape}; ¿labels = ATTACK_TYPES?: {attack_type['confusion']['labels'] == ATTACK_TYPES}")

nothing = attack_type_metrics(["Generic", "Normal"], [None, None], [False, False])
print(f"\nSin flujos evaluados: evaluated_flows = {nothing['evaluated_flows']}, macro_f1 = {nothing['macro_f1']}")
print(f"¿Todo se puede escribir en JSON?: {bool(json.dumps(attack_type)) and bool(json.dumps(nothing))}")

print()
for name, bad_true, bad_pred, bad_alerts in (
    ("Tipo real que no es de los 9", ["Generic", "worms"], ["Generic", "Worms"], [True, True]),
    ("Alerta con tipo probable Normal", ["Generic", "DoS"], ["Generic", "Normal"], [True, True]),
    ("Distinta cantidad", ["Generic", "DoS"], ["Generic"], [True, True]),
):
    try:
        attack_type_metrics(bad_true, bad_pred, bad_alerts)
    except ValueError as error:
        print(f"{name}: {error}")

matrix

evaluated_flows = 8; tp de detection_metrics = 8 (esperados 8)
¿Métricas de los 9 tipos como se esperaba?: True; ¿los 9 en orden?: True
macro_f1 = 0.4405 (esperado 0.4405)
Matriz: (9, 9); ¿labels = ATTACK_TYPES?: True

Sin flujos evaluados: evaluated_flows = 0, macro_f1 = None
¿Todo se puede escribir en JSON?: True

Tipo real que no es de los 9: Hay ataques reales con un tipo que no es uno de los 9 tipos de ataque: worms.
Alerta con tipo probable Normal: Hay alertas con un tipo probable que no es uno de los 9 tipos de ataque: Normal.
Distinta cantidad: Los tipos reales traen 2 flujos, los predichos 1 y las alertas 2; deben traer los mismos flujos.


,Generic,Exploits,Fuzzers,DoS,Reconnaissance,Analysis,Backdoor,Shellcode,Worms
Generic,3,1,0,0,0,0,0,0,0
Exploits,1,2,0,0,0,0,0,0,0
Fuzzers,0,0,0,0,0,0,0,0,0
DoS,0,1,0,0,0,0,0,0,0
Reconnaissance,0,0,0,0,0,0,0,0,0
Analysis,0,0,0,0,0,0,0,0,0
Backdoor,0,0,0,0,0,0,0,0,0
Shellcode,0,0,0,0,0,0,0,0,0
Worms,0,0,0,0,0,0,0,0,0


## 8. Formato del contrato

Las secciones anteriores comprobaron cada función por separado. Aquí comprobamos el conjunto contra **el contrato mismo**: armamos `threshold_rule` y `test_metrics` (`test_flows`, `detection` y `attack_type`, como en `model.json`) y los comparamos con el ejemplo de `model.json` que trae `docs/contracts.md` (sección 11).

Primero leemos todos los ejemplos JSON del contrato. Si alguno no se puede interpretar (por ejemplo, porque tiene comentarios o puntos suspensivos), la sección lo dice e indica cuál es y en qué sección del contrato está.

Después comparamos la **estructura** de forma recursiva: las mismas claves, en el mismo orden, en cada nivel (y una entrada por cada tipo de ataque en `per_class`). Los valores no se comparan, porque los del contrato son inventados. Por último, comprobamos que el resultado se puede escribir en JSON y leerse de vuelta sin cambios, es decir, que no quedan tipos de numpy.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos los resultados con datos escritos a mano de las secciones 5 y 7. `test_flows` es la cantidad de flujos de esos datos; con un modelo entrenado, es la cantidad de flujos de la prueba.

In [9]:
import re

from threat_detector.config import ROOT

contract_text = (ROOT / "docs" / "contracts.md").read_text(encoding="utf-8")

# Cada ejemplo JSON del contrato, con la sección (encabezado "## ") en la que está.
examples = []
section = None
for match in re.finditer(r"^## ([^\n]+)$|^```json\n(.*?)^```", contract_text, re.MULTILINE | re.DOTALL):
    if match.group(1):
        section = match.group(1)
    else:
        examples.append((section, match.group(2)))

parsed = []
for number, (section, block) in enumerate(examples, start=1):
    try:
        parsed.append((section, json.loads(block)))
    except json.JSONDecodeError as error:
        print(f"No se pudo interpretar el ejemplo JSON n.º {number} (sección «{section}»): {error}")
print(f"Ejemplos JSON del contrato: {len(examples)}; interpretados: {len(parsed)}")

model_examples = [example for _, example in parsed if "test_metrics" in example and "threshold_rule" in example]
if not model_examples:
    raise ValueError("No se encontró en contracts.md el ejemplo de model.json (con test_metrics y threshold_rule).")
model_json = model_examples[0]


def structure(value):
    """Estructura de un valor JSON: claves y orden de los objetos, sin los valores."""
    if isinstance(value, dict):
        return [(key, structure(item)) for key, item in value.items()]
    if isinstance(value, list) and value and isinstance(value[0], dict):
        return [structure(item) for item in value]
    return "valor"


test_metrics = {"test_flows": len(type_labels), "detection": type_detection, "attack_type": attack_type}
for name, ours, theirs in (
    ("threshold_rule", rule, model_json["threshold_rule"]),
    ("test_metrics", test_metrics, model_json["test_metrics"]),
):
    print(f"{name}: ¿misma estructura que el contrato?: {structure(ours) == structure(theirs)}; "
          f"¿JSON de ida y vuelta sin cambios?: {json.loads(json.dumps(ours)) == ours}")

print(f"¿evaluated_flows = tp?: {attack_type['evaluated_flows'] == type_detection['confusion']['tp']}")

Ejemplos JSON del contrato: 18; interpretados: 18
threshold_rule: ¿misma estructura que el contrato?: True; ¿JSON de ida y vuelta sin cambios?: True
test_metrics: ¿misma estructura que el contrato?: True; ¿JSON de ida y vuelta sin cambios?: True
¿evaluated_flows = tp?: True


## 9. Clases por su nombre

`predict_proba` devuelve una columna por clase, en el orden de `classes_` del modelo. Ese orden lo fija el modelo al entrenarse, así que las funciones de `artifacts.py` buscan cada clase **por su nombre**, nunca por su posición (contrato, sección 10):

- `attack_probability_from_proba`: con un detector binario (`detector_type` = `"binary"`), la probabilidad de ataque es la columna de la clase `1`; con uno multiclase (`"multiclass"`), es 1 menos la columna de `"Normal"`. `detector_type` sale de `model.json`; nunca se deduce.
- `attack_cat_from_proba`: el tipo más probable entre los 9 de `ATTACK_TYPES`. La columna de `"Normal"` no se mira aunque el clasificador la tenga, para que una alerta nunca diga "tipo probable: Normal". Las columnas se ordenan como `ATTACK_TYPES`, así que en un empate gana el primero en ese orden, sin importar el orden de las clases.

> **Datos inventados.** Como aún no hay modelos entrenados, usamos probabilidades escritas a mano y las presentamos con las clases en distinto orden. Así comprobamos que el resultado no depende del orden de `classes_`.

Flujos del caso multiclase (las clases que no aparecen tienen probabilidad 0):

| Flujo | Probabilidades | Probabilidad de ataque esperada | Tipo esperado |
|---|---|---|---|
| 1 | Normal 0,70; Generic 0,20; DoS 0,10 | 1 − 0,70 = 0,30 | Generic: `"Normal"` es la más probable, pero no se mira |
| 2 | Normal 0,10; Exploits 0,50; Fuzzers 0,40 | 1 − 0,10 = 0,90 | Exploits |
| 3 | Normal 0,05; Exploits 0,40; DoS 0,40; Worms 0,15 | 1 − 0,05 = 0,95 | Exploits: empata con DoS y va antes en `ATTACK_TYPES` |

**Resultado esperado:**
- Detector binario con las clases `[0, 1]` y `[1, 0]` (columnas intercambiadas): la misma probabilidad, igual a la columna de la clase `1`.
- Detector multiclase con las 10 clases en orden alfabético y en el orden inverso: la misma probabilidad, igual a 1 − la columna de `"Normal"` (tabla).
- Tipo probable con las 10 clases en los dos órdenes y con un clasificador que solo tiene los 9 tipos (sin `"Normal"`): Generic, Exploits y Exploits; nunca `"Normal"`.
- Errores en español: detector binario sin la clase `1`, detector multiclase sin `"Normal"`, clasificador al que le falta un tipo, `detector_type` desconocido y probabilidades con un número de columnas distinto al de las clases.

In [10]:
from threat_detector.artifacts import attack_cat_from_proba, attack_probability_from_proba
from threat_detector.dataset import ATTACK_TYPES, NORMAL_CAT

# Caso 1: detector binario con 3 flujos. Se arma con las clases [0, 1] y se invierte para [1, 0].
attack_column = np.array([0.80, 0.30, 0.05])
binary_proba = np.column_stack([1 - attack_column, attack_column])
for classes, proba in (([0, 1], binary_proba), ([1, 0], binary_proba[:, ::-1])):
    result = attack_probability_from_proba(proba, np.array(classes), "binary")
    print(f"Binario, clases {classes}: {result}; ¿igual a la columna de 1?: {np.array_equal(result, attack_column)}")


def to_proba(rows, classes):
    """Matriz de probabilidades con una columna por clase, en el orden de classes (0 si no aparece)."""
    return np.array([[row.get(c, 0.0) for c in classes] for row in rows])


# Caso 2: los 3 flujos de la tabla, con las 10 clases en dos órdenes distintos.
rows = [
    {"Normal": 0.70, "Generic": 0.20, "DoS": 0.10},
    {"Normal": 0.10, "Exploits": 0.50, "Fuzzers": 0.40},
    {"Normal": 0.05, "Exploits": 0.40, "DoS": 0.40, "Worms": 0.15},
]
expected_probability = 1 - np.array([0.70, 0.10, 0.05])
expected_types = ["Generic", "Exploits", "Exploits"]
alphabetical = sorted([NORMAL_CAT, *ATTACK_TYPES])
orders = {"alfabético": alphabetical, "inverso": alphabetical[::-1]}
print()
for order_name, classes in orders.items():
    proba = to_proba(rows, classes)
    probability = attack_probability_from_proba(proba, np.array(classes, dtype=object), "multiclass")
    types = attack_cat_from_proba(proba, np.array(classes, dtype=object))
    print(f"Multiclase, orden {order_name}: probabilidad {probability} "
          f"(¿esperada?: {np.array_equal(probability, expected_probability)}); "
          f"tipos {list(types)} (¿esperados?: {list(types) == expected_types})")

# Caso 3: clasificador con solo los 9 tipos (sin "Normal"), en orden inverso. Cada fila se
# reparte entre los 9 tipos para que sume 1, como la salida de un clasificador real.
classes = ATTACK_TYPES[::-1]
proba = to_proba(rows, classes)
proba = proba / proba.sum(axis=1, keepdims=True)
types = attack_cat_from_proba(proba, np.array(classes, dtype=object))
print(f"\nClasificador sin Normal: tipos {list(types)} (¿esperados?: {list(types) == expected_types})")
print(f"¿Algún tipo probable es Normal?: {NORMAL_CAT in types}")

# Caso 4: errores, a propósito.
print()
nine_classes = np.array(ATTACK_TYPES, dtype=object)
missing_worms = np.array([c for c in alphabetical if c != "Worms"], dtype=object)
error_cases = (
    ("Binario sin la clase 1", lambda: attack_probability_from_proba(binary_proba, np.array([0, 2]), "binary")),
    ("Multiclase sin Normal", lambda: attack_probability_from_proba(to_proba(rows, nine_classes), nine_classes, "multiclass")),
    ("Clasificador sin Worms", lambda: attack_cat_from_proba(to_proba(rows, missing_worms), missing_worms)),
    ("detector_type desconocido", lambda: attack_probability_from_proba(binary_proba, np.array([0, 1]), "binario")),
    ("Columnas distintas a las clases", lambda: attack_probability_from_proba(binary_proba, np.array([0, 1, 2]), "binary")),
)
for name, call in error_cases:
    try:
        call()
        print(f"{name}: no lanzó error")
    except ValueError as error:
        print(f"{name}: {error}")

Binario, clases [0, 1]: [0.8  0.3  0.05]; ¿igual a la columna de 1?: True
Binario, clases [1, 0]: [0.8  0.3  0.05]; ¿igual a la columna de 1?: True

Multiclase, orden alfabético: probabilidad [0.3  0.9  0.95] (¿esperada?: True); tipos ['Generic', 'Exploits', 'Exploits'] (¿esperados?: True)
Multiclase, orden inverso: probabilidad [0.3  0.9  0.95] (¿esperada?: True); tipos ['Generic', 'Exploits', 'Exploits'] (¿esperados?: True)

Clasificador sin Normal: tipos ['Generic', 'Exploits', 'Exploits'] (¿esperados?: True)
¿Algún tipo probable es Normal?: False

Binario sin la clase 1: El modelo no tiene la clase 1 entre sus clases: 0, 2.
Multiclase sin Normal: El modelo no tiene la clase Normal entre sus clases: Generic, Exploits, Fuzzers, DoS, Reconnaissance, Analysis, Backdoor, Shellcode, Worms.
Clasificador sin Worms: El modelo no tiene la clase Worms entre sus clases: Analysis, Backdoor, DoS, Exploits, Fuzzers, Generic, Normal, Reconnaissance, Shellcode.
detector_type desconocido: Tipo de de

## 10. Modelos de prueba y predicción de flujos

`predict_flows` (en `artifacts.py`) es lo que hará la app con cada archivo: recibe el modelo (el diccionario de `model.joblib`, con `"detector"` y `"attack_type_classifier"`), su `model.json` y los flujos ya validados, y devuelve tres arreglos con un valor por flujo:

1. `attack_probability`: con `predict_proba` del detector y `attack_probability_from_proba` (sección 9), según `detector_type`.
2. `alerts`: con `is_alert` y el umbral del modelo.
3. `predicted_attack_cat`: con `predict_proba` del clasificador y `attack_cat_from_proba`, **solo para las alertas**; en los demás flujos es `None` (contrato, sección 3). Si el detector y el clasificador son el mismo `Pipeline`, reutiliza las probabilidades del detector en vez de volver a calcularlas, y si no hay ninguna alerta, el clasificador no se usa.

> **Modelos de prueba.** Como aún no hay modelos del proyecto, entrenamos aquí modelos de prueba muy pequeños, solo para tener una salida real de `predict_proba` con la que probar las funciones. **No son los modelos del proyecto, no se eligieron por su rendimiento y no se registran.** Se entrenan con 100 flujos de cada valor de `attack_cat`, tomados al azar (con `RANDOM_STATE`) de la parte de entrenamiento, y se prueban con 2.000 flujos de la parte de validación. La prueba no se usa.

Cada `Pipeline` tiene dos pasos con nombre fijo: `"preprocessor"` (`build_preprocessor` con un método) y `"model"` (el estimador). Probamos las dos formas que permite el contrato (sección 10):

| Forma | `detector` | `attack_type_classifier` | `detector_type` |
|---|---|---|---|
| A con 9 clases | Regresión logística sobre `label` (M2) | Árbol de decisión sobre `attack_cat`, entrenado solo con ataques (9 clases, M2); su preprocesador se ajusta con todos los flujos, como pide el contrato (sección 10), para conservar las mismas categorías que el detector | `"binary"` |
| A con 10 clases | El mismo detector | Árbol de decisión sobre `attack_cat`, entrenado con todos los flujos, incluido `"Normal"` (10 clases, M2) | `"binary"` |
| B | Un solo árbol de decisión sobre `attack_cat` (10 clases, M3) | El mismo `Pipeline` | `"multiclass"` |

Cuál de las dos maneras de entrenar el clasificador (9 o 10 clases) se usa se decide al elegir los modelos; aquí se prueban las dos.

El umbral de cada forma se calcula con `compute_threshold` sobre los mismos 2.000 flujos, para que haya alertas y flujos sin alerta. Como `build_model_info` todavía no existe, a `predict_flows` le pasamos un `model.json` mínimo con solo los dos campos que usa: `detector_type` y `threshold`. Los flujos se pasan con todas sus columnas, como llegarían en un archivo con etiquetas; el preprocesador solo usa las variables de entrada.

**Resultado esperado**, en las tres formas:
- `classes_` de cada modelo, que muestra en qué orden guarda scikit-learn las clases.
- Probabilidades entre 0 y 1, iguales a la columna de la clase `1` (forma A) o a 1 − la columna de `"Normal"` (forma B) de `predict_proba`.
- `alerts` igual a probabilidad ≥ umbral.
- `predicted_attack_cat` es `None` exactamente en los flujos sin alerta; en las alertas es uno de los 9 tipos, nunca `"Normal"`, y es el tipo con la mayor probabilidad entre los 9.
- En las formas con 10 clases, cuántas alertas tienen `"Normal"` como clase más probable del clasificador: en esas, el tipo probable igual es uno de los 9.
- Llamadas a `predict_proba`: 1 al detector y 1 al clasificador en la forma A; 1 en total en la forma B; 0 al clasificador si no hay alertas (umbral 1,01).
- El resultado se puede escribir en JSON flujo por flujo, sin tipos de numpy.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from threat_detector.artifacts import predict_flows
from threat_detector.config import RANDOM_STATE
from threat_detector.ml.preprocessing import build_preprocessor, split_train_validation

TEST_MODEL_FLOWS_PER_CLASS = 100   # flujos de cada valor de attack_cat para entrenar los modelos de prueba
TEST_MODEL_VALIDATION_FLOWS = 2000  # flujos de validación con los que se prueban
TEST_TREE_DEPTH = 5                 # profundidad de los árboles de prueba, para que den probabilidades intermedias

train_part, validation_part = split_train_validation(load_train())
small_train = train_part.groupby(ATTACK_CAT).sample(n=TEST_MODEL_FLOWS_PER_CLASS, random_state=RANDOM_STATE)
small_attacks = small_train[small_train[TARGET] == 1]
test_flows = validation_part.sample(n=TEST_MODEL_VALIDATION_FLOWS, random_state=RANDOM_STATE)
print(f"Entrenamiento de prueba: {len(small_train)} flujos ({len(small_attacks)} ataques); "
      f"validación de prueba: {len(test_flows)} flujos ({test_flows[TARGET].sum()} ataques)")


def test_pipeline(method, estimator):
    """Pipeline de prueba con los dos pasos de nombre fijo."""
    return Pipeline([("preprocessor", build_preprocessor(method)), ("model", estimator)])


def tree():
    return DecisionTreeClassifier(max_depth=TEST_TREE_DEPTH, random_state=RANDOM_STATE)


def attack_pipeline(method, estimator, flows):
    """Clasificador de prueba como pide el contrato (sección 10): el preprocesador se ajusta con todos
    los flujos y el estimador aprende solo con los ataques. Ajustado solo con ataques, perdería
    categorías que solo tienen los flujos normales (aquí, proto_arp y service_ftp-data)."""
    attacks = flows[flows[TARGET] == 1]
    preprocessor = build_preprocessor(method).fit(flows)
    estimator.fit(preprocessor.transform(attacks), attacks[ATTACK_CAT])
    return Pipeline([("preprocessor", preprocessor), ("model", estimator)])


detector_a = test_pipeline("m2", LogisticRegression(max_iter=1000)).fit(small_train, small_train[TARGET])
classifier_9 = attack_pipeline("m2", tree(), small_train)
classifier_10 = test_pipeline("m2", tree()).fit(small_train, small_train[ATTACK_CAT])
multiclass_b = test_pipeline("m3", tree()).fit(small_train, small_train[ATTACK_CAT])

test_models = {
    "A con 9 clases": ({"detector": detector_a, "attack_type_classifier": classifier_9}, "binary"),
    "A con 10 clases": ({"detector": detector_a, "attack_type_classifier": classifier_10}, "binary"),
    "B": ({"detector": multiclass_b, "attack_type_classifier": multiclass_b}, "multiclass"),
}

print("\nclasses_ de cada modelo de prueba:")
for name, pipeline in (("detector A", detector_a), ("clasificador 9", classifier_9),
                       ("clasificador 10", classifier_10), ("multiclase B", multiclass_b)):
    classes = list(pipeline.classes_)
    print(f"  {name}: {', '.join(map(str, classes))}; ¿ordenadas?: {classes == sorted(classes)}")

Entrenamiento de prueba: 1000 flujos (900 ataques); validación de prueba: 2000 flujos (1372 ataques)

classes_ de cada modelo de prueba:
  detector A: 0, 1; ¿ordenadas?: True
  clasificador 9: Analysis, Backdoor, DoS, Exploits, Fuzzers, Generic, Reconnaissance, Shellcode, Worms; ¿ordenadas?: True
  clasificador 10: Analysis, Backdoor, DoS, Exploits, Fuzzers, Generic, Normal, Reconnaissance, Shellcode, Worms; ¿ordenadas?: True
  multiclase B: Analysis, Backdoor, DoS, Exploits, Fuzzers, Generic, Normal, Reconnaissance, Shellcode, Worms; ¿ordenadas?: True


Con los modelos de prueba entrenados, predecimos los 2.000 flujos con `predict_flows` en cada forma y comparamos el resultado con lo que se calcula a mano desde `predict_proba`. Para contar las llamadas a `predict_proba`, cada `Pipeline` va dentro de un envoltorio de prueba (`CountingModel`) que hace lo mismo y lleva la cuenta.

In [12]:
class CountingModel:
    """Envoltorio de prueba: se comporta como el Pipeline y cuenta las llamadas a predict_proba."""

    def __init__(self, pipeline):
        self.pipeline = pipeline
        self.classes_ = pipeline.classes_
        self.calls = 0

    def predict_proba(self, flows):
        self.calls += 1
        return self.pipeline.predict_proba(flows)


test_infos = {}
for name, (model, detector_type) in test_models.items():
    detector, classifier = model["detector"], model["attack_type_classifier"]
    detector_proba = detector.predict_proba(test_flows)
    if detector_type == "binary":
        expected = detector_proba[:, list(detector.classes_).index(1)]
    else:
        expected = 1 - detector_proba[:, list(detector.classes_).index(NORMAL_CAT)]
    info = {"detector_type": detector_type, "threshold": compute_threshold(test_flows[TARGET], expected)}
    test_infos[name] = info

    # Con envoltorios que cuentan las llamadas; si es el mismo Pipeline, el mismo envoltorio en las dos claves.
    counting_detector = CountingModel(detector)
    counting_classifier = counting_detector if classifier is detector else CountingModel(classifier)
    probability, alerts, types = predict_flows(
        {"detector": counting_detector, "attack_type_classifier": counting_classifier}, info, test_flows
    )

    no_type = np.array([t is None for t in types])
    alert_types = types[alerts]
    # Probabilidades del clasificador en las alertas, con las columnas en el orden de ATTACK_TYPES.
    type_proba = classifier.predict_proba(test_flows[alerts])
    type_columns = [list(classifier.classes_).index(t) for t in ATTACK_TYPES]
    chosen = np.array([list(classifier.classes_).index(t) for t in alert_types])
    has_max = np.allclose(type_proba[np.arange(len(chosen)), chosen], type_proba[:, type_columns].max(axis=1))

    print(f"Forma {name}: umbral {info['threshold']:.4f}; {alerts.sum()} alertas de {len(alerts)} flujos")
    print(f"  ¿Probabilidades entre 0 y 1?: {bool(((probability >= 0) & (probability <= 1)).all())}; "
          f"¿iguales a predict_proba?: {np.array_equal(probability, expected)}; "
          f"¿alertas = probabilidad ≥ umbral?: {np.array_equal(alerts, probability >= info['threshold'])}")
    print(f"  ¿Tipo None exactamente sin alerta?: {np.array_equal(no_type, ~alerts)}; "
          f"¿todos los tipos de las alertas entre los 9?: {set(alert_types) <= set(ATTACK_TYPES)}; "
          f"¿alguno Normal?: {NORMAL_CAT in alert_types}; ¿el de mayor probabilidad entre los 9?: {has_max}")
    if NORMAL_CAT in classifier.classes_:
        normal_top = (classifier.predict(test_flows[alerts]) == NORMAL_CAT).sum()
        print(f"  Alertas con Normal como clase más probable del clasificador: {normal_top}")
    if classifier is detector:
        print(f"  Llamadas a predict_proba: {counting_detector.calls} en total (esperada 1)")
    else:
        print(f"  Llamadas a predict_proba: detector {counting_detector.calls}, "
              f"clasificador {counting_classifier.calls} (esperadas 1 y 1)")

# Sin ninguna alerta: umbral 1,01, que ninguna probabilidad alcanza.
model, _ = test_models["A con 9 clases"]
counting_detector, counting_classifier = CountingModel(model["detector"]), CountingModel(model["attack_type_classifier"])
_, alerts, types = predict_flows(
    {"detector": counting_detector, "attack_type_classifier": counting_classifier},
    {"detector_type": "binary", "threshold": 1.01}, test_flows,
)
print(f"\nSin alertas: {alerts.sum()} alertas; ¿todos los tipos None?: {all(t is None for t in types)}; "
      f"llamadas al clasificador: {counting_classifier.calls} (esperada 0)")

# Cada flujo como en el JSON del contrato, con .tolist() para tener tipos de Python.
model, _ = test_models["A con 9 clases"]
probability, alerts, types = predict_flows(model, test_infos["A con 9 clases"], test_flows)
flows_json = [
    {"id": int(i), "attack_probability": p, "is_alert": a, "predicted_attack_cat": t}
    for i, p, a, t in zip(test_flows["id"], probability.tolist(), alerts.tolist(), types.tolist())
]
print(f"¿Se puede escribir en JSON y leer de vuelta sin cambios?: {json.loads(json.dumps(flows_json)) == flows_json}")
pd.DataFrame(flows_json).head()

Forma A con 9 clases: umbral 0.8421; 1405 alertas de 2000 flujos
  ¿Probabilidades entre 0 y 1?: True; ¿iguales a predict_proba?: True; ¿alertas = probabilidad ≥ umbral?: True
  ¿Tipo None exactamente sin alerta?: True; ¿todos los tipos de las alertas entre los 9?: True; ¿alguno Normal?: False; ¿el de mayor probabilidad entre los 9?: True
  Llamadas a predict_proba: detector 1, clasificador 1 (esperadas 1 y 1)
Forma A con 10 clases: umbral 0.8421; 1405 alertas de 2000 flujos
  ¿Probabilidades entre 0 y 1?: True; ¿iguales a predict_proba?: True; ¿alertas = probabilidad ≥ umbral?: True
  ¿Tipo None exactamente sin alerta?: True; ¿todos los tipos de las alertas entre los 9?: True; ¿alguno Normal?: False; ¿el de mayor probabilidad entre los 9?: True
  Alertas con Normal como clase más probable del clasificador: 1
  Llamadas a predict_proba: detector 1, clasificador 1 (esperadas 1 y 1)
Forma B: umbral 0.8788; 1553 alertas de 2000 flujos
  ¿Probabilidades entre 0 y 1?: True; ¿iguales a predi

,id,attack_probability,is_alert,predicted_attack_cat
0,143469,0.999992,True,Generic
1,170782,0.996358,True,Fuzzers
2,160277,0.999839,True,Generic
3,7234,0.006363,False,NaN
4,134483,0.999900,True,Generic


## 11. Descripción del modelo (`model.json`)

`ml/training.py` arma `model.json` con dos funciones:

- `feature_groups`: para cada variable original, las columnas que salen del preprocesador ya ajustado (los nombres de `get_feature_names_out`). La app lo usará para sumar las contribuciones por variable. Las claves van en el orden del contrato: primero las numéricas, en el orden del archivo, y después `proto`, `service` y `state`.
- `build_model_info`: devuelve el diccionario de `model.json` con todos sus campos, en el orden del contrato (sección 11). Calcula por su cuenta `is_mock`, `sklearn_version`, `use_ttl`, las listas de variables y `feature_groups`; lo demás lo recibe. `trained_at` se recibe con zona horaria y se escribe en UTC. `model_size_mb` queda en `None` hasta que el modelo se guarde.

Antes de armarlo, `build_model_info` comprueba que cada `Pipeline` (detector y clasificador) tiene exactamente los pasos `"preprocessor"` y `"model"`, y que su preprocesador tiene la misma estructura que `build_preprocessor` con el método declarado: las mismas partes, con las mismas piezas y las mismas columnas. No basta con comparar los nombres de las columnas de salida, porque M1 y M2 dan los mismos nombres; solo cambia el `StandardScaler`. Así se comprueba a la vez que el método de `model.json` es el verdadero y que el detector y el clasificador usan el mismo.

> **Modelos de prueba y valores inventados.** Usamos los modelos de prueba de la sección 10, que no son modelos del proyecto. El nombre, la fecha de entrenamiento y `complexity` son inventados. `threshold_rule` y `test_metrics` se calculan con las funciones de `evaluation.py`, pero sobre los 2.000 flujos de validación de la sección 10, no sobre la prueba.

Las categorías que conserva el one-hot dependen de los flujos con que se ajusta el preprocesador, porque `MIN_CATEGORY_FRACTION` es una proporción. Los modelos de prueba se entrenaron con solo 1.000 flujos, así que, para comparar `feature_groups` con el contrato, ajustamos además solo el preprocesador (sin modelo) con toda la parte de entrenamiento, como se ajustará en los modelos reales.

**Resultado esperado:**
- `feature_groups` de un preprocesador M2 ajustado con toda la parte de entrenamiento: idéntico, en contenido y en orden, al del ejemplo de `model.json` del contrato, que tiene las listas reales de las decisiones del análisis exploratorio.
- `feature_groups` de un preprocesador M3 ajustado igual: 29 numéricas, ninguna de TTL, y cada columna de salida en exactamente un grupo.
- `feature_groups` del detector de prueba de la forma A: las mismas variables, pero otras categorías, porque se ajustó con 1.000 flujos. Se muestran las diferencias y las categorías que quedaron como infrecuentes en cada variable.
- `model.json` de la forma A: la misma estructura que el ejemplo del contrato (con la función `structure` de la sección 8), `numeric_cols` y `log_cols` iguales a las del ejemplo, `use_ttl` igual a `True` (M2), la versión de scikit-learn instalada, `model_size_mb` en `None` y JSON de ida y vuelta sin cambios.
- `trained_at`: el 20 de noviembre de 2026 a las 10:42:10 en Colombia (UTC−5) se escribe `"2026-11-20T15:42:10Z"`.
- Errores en español, a propósito: la forma A declarada como M1, un detector M2 con un clasificador M3, un `Pipeline` hecho con `make_pipeline` (sus pasos no se llaman `"preprocessor"` y `"model"`), un `detector_type` desconocido, un método desconocido y una fecha sin zona horaria.

In [13]:
from datetime import datetime, timedelta, timezone

import sklearn
from sklearn.pipeline import make_pipeline

from threat_detector.dataset import CATEGORICAL_COLS
from threat_detector.ml.evaluation import attack_type_metrics
from threat_detector.ml.preprocessing import TTL_COLS
from threat_detector.ml.training import build_model_info, feature_groups

# feature_groups con preprocesadores M2 y M3 ajustados con toda la parte de entrenamiento (sin modelo).
preprocessor_m2 = build_preprocessor("m2").fit(train_part)
groups_m2 = feature_groups(preprocessor_m2)
print(f"M2, toda la parte de entrenamiento: {len(groups_m2)} variables; ¿igual al ejemplo del contrato, "
      f"en contenido y orden?: {list(groups_m2.items()) == list(model_json['feature_groups'].items())}")
preprocessor_m3 = build_preprocessor("m3").fit(train_part)
groups_m3 = feature_groups(preprocessor_m3)
numeric_m3 = [c for c in groups_m3 if c not in CATEGORICAL_COLS]
grouped_m3 = [name for names in groups_m3.values() for name in names]
print(f"M3, toda la parte de entrenamiento: {len(numeric_m3)} numéricas; "
      f"¿alguna de TTL?: {any(c in groups_m3 for c in TTL_COLS)}; ¿cada columna en exactamente un grupo?: "
      f"{sorted(grouped_m3) == sorted(preprocessor_m3.get_feature_names_out())}")

# El detector de prueba de la forma A se ajustó con 1.000 flujos: mismas variables, otras categorías.
groups_a = feature_groups(detector_a["preprocessor"])
print(f"\nDetector de prueba A: ¿mismas variables que el contrato?: {list(groups_a) == list(model_json['feature_groups'])}")
encoder_a = detector_a["preprocessor"].named_transformers_["categorical"]
for col, infrequent in zip(CATEGORICAL_COLS, encoder_a.infrequent_categories_):
    extra = [c for c in groups_a[col] if c not in groups_m2[col]]
    missing = [c for c in groups_m2[col] if c not in groups_a[col]]
    print(f"  {col}: de más {extra}, faltan {missing}; infrecuentes al ajustar: "
          f"{0 if infrequent is None else len(infrequent)}")

# model.json de la forma A, con los modelos de prueba y valores inventados.
model_a = test_models["A con 9 clases"][0]
threshold_a = test_infos["A con 9 clases"]["threshold"]
probability, alerts, types = predict_flows(model_a, test_infos["A con 9 clases"], test_flows)
base_args = {
    "name": "test_model_a",
    "display_name": "Modelo de prueba A",
    "model": model_a,
    "detector_type": "binary",
    "preprocessing_method": "m2",
    "threshold": threshold_a,
    "threshold_rule": threshold_rule(test_flows[TARGET], probability, threshold_a),
    "can_explain": False,
    "explanation_method": None,
    "test_metrics": {
        "test_flows": len(test_flows),
        "detection": detection_metrics(test_flows[TARGET], probability, threshold_a),
        "attack_type": attack_type_metrics(test_flows[ATTACK_CAT], types, alerts),
    },
    "complexity": {"training_seconds": 1.0, "training_flows": len(small_train),
                   "prediction_ms_per_1000_flows": 1.0, "hardware": "Equipo de prueba"},
    # 10:42:10 en Colombia (UTC−5).
    "trained_at": datetime(2026, 11, 20, 10, 42, 10, tzinfo=timezone(timedelta(hours=-5))),
}
info_a = build_model_info(**base_args)

print(f"\n¿Misma estructura que el ejemplo de model.json?: {structure(info_a) == structure(model_json)}")
print(f"¿numeric_cols y log_cols iguales a las del ejemplo?: "
      f"{info_a['numeric_cols'] == model_json['numeric_cols'] and info_a['log_cols'] == model_json['log_cols']}")
print(f"use_ttl = {info_a['use_ttl']} con {info_a['preprocessing_method']}; "
      f"sklearn_version = {info_a['sklearn_version']} (instalada: {sklearn.__version__}); "
      f"model_size_mb = {info_a['complexity']['model_size_mb']}")
print(f"trained_at = {info_a['trained_at']} (esperado 2026-11-20T15:42:10Z)")
print(f"¿JSON de ida y vuelta sin cambios?: {json.loads(json.dumps(info_a)) == info_a}")


def build_with(**changes):
    """build_model_info con los argumentos de la forma A, cambiando solo los indicados."""
    return build_model_info(**{**base_args, **changes})


print()
error_cases = (
    ("Forma A declarada como M1", {"preprocessing_method": "m1"}),
    ("Detector M2 con clasificador M3", {"model": {"detector": detector_a, "attack_type_classifier": multiclass_b}}),
    ("Pipeline con make_pipeline", {"model": {"detector": make_pipeline(build_preprocessor("m2"), LogisticRegression()),
                                              "attack_type_classifier": classifier_9}}),
    ("detector_type desconocido", {"detector_type": "binario"}),
    ("Método desconocido", {"preprocessing_method": "m4"}),
    ("Fecha sin zona horaria", {"trained_at": datetime(2026, 11, 20, 15, 42, 10)}),
)
for name, changes in error_cases:
    try:
        build_with(**changes)
        print(f"{name}: no lanzó error")
    except ValueError as error:
        print(f"{name}: {error}")

M2, toda la parte de entrenamiento: 35 variables; ¿igual al ejemplo del contrato, en contenido y orden?: True
M3, toda la parte de entrenamiento: 29 numéricas; ¿alguna de TTL?: False; ¿cada columna en exactamente un grupo?: True

Detector de prueba A: ¿mismas variables que el contrato?: True
  proto: de más ['proto_sun-nd'], faltan []; infrecuentes al ajustar: 74
  service: de más [], faltan ['service_pop3', 'service_ssh']; infrecuentes al ajustar: 4
  state: de más [], faltan ['state_infrequent_sklearn']; infrecuentes al ajustar: 0

¿Misma estructura que el ejemplo de model.json?: True
¿numeric_cols y log_cols iguales a las del ejemplo?: True
use_ttl = True con m2; sklearn_version = 1.9.1 (instalada: 1.9.1); model_size_mb = None
trained_at = 2026-11-20T15:42:10Z (esperado 2026-11-20T15:42:10Z)
¿JSON de ida y vuelta sin cambios?: True

Forma A declarada como M1: El preprocesador de detector no es el del método m1: el detector y el clasificador deben usar el método declarado.
Detector M

## 12. Exportar un modelo

`export_model` (en `ml/training.py`) guarda un modelo en `artifacts/models/<name>/` con sus dos archivos (contrato, sección 10):

1. Comprueba que `name` está en minúsculas y `snake_case` (es el nombre de la carpeta) y no es `"mock"`, que es el modelo simulado y se escribe a mano. Después aplica al modelo y a su `model.json` las mismas comprobaciones que hace la app al cargarlo (secciones 13 y 14), con `test_metrics` opcional, para que un modelo exportado siempre se pueda cargar.
2. Comprueba que `model.json` se puede escribir como JSON válido **antes de escribir cualquier archivo**. Así, si falla (por ejemplo, por un `NaN` en una métrica), la carpeta no queda con el `model.joblib` nuevo y el `model.json` viejo.
3. Guarda `model.joblib` con `joblib`, sin compresión, y mide `model_size_mb`: el tamaño en disco dividido entre 1.000.000.
4. Con `check_model_size`, avisa (sin error) si el modelo pesa `MAX_MODEL_SIZE_MB` (50 MB) o más, para que el equipo lo revise antes de subirlo a git.
5. Escribe `model.json` en UTF-8, con las tildes legibles, y devuelve lo que escribió. Si la carpeta ya existía, los dos archivos se reemplazan. El modelo no queda registrado.

> **Modelos de prueba en una carpeta temporal.** Exportamos los modelos de prueba de la sección 10, que no son modelos del proyecto, a una carpeta temporal en lugar de `artifacts/`. La carpeta se borra al final del notebook. Para comprobarlo, tomamos una foto de los archivos de `artifacts/` del proyecto (ruta, tamaño y fecha de modificación) antes y después de esta sección.

Para la forma B armamos su `model.json` con su propio umbral, `threshold_rule` y métricas, calculados como los de la forma A en la sección 11 (sobre los 2.000 flujos de validación, no sobre la prueba).

**Resultado esperado:**
- Formas A y B: la carpeta tiene exactamente `model.joblib` y `model.json`; `model_size_mb` es igual al tamaño en disco ÷ 1.000.000 y menor que 50, sin aviso; `model.json` leído de vuelta es igual a lo que devolvió la función, y el `info` recibido no cambió (`model_size_mb` sigue en `None`).
- Las tildes de `display_name` se leen tal cual en el archivo, sin escapar.
- `check_model_size`: sin aviso con 49,99 MB; con aviso con 50 y 50,01 MB.
- Exportar de nuevo la forma A con otro umbral: `model.json` trae el umbral nuevo y la carpeta sigue con dos archivos.
- Un `NaN` en `test_metrics`, exportado con el modelo de la forma A con 10 clases: error en español, y los dos archivos de la carpeta quedan idénticos (mismo contenido y misma fecha de modificación) a como estaban antes del intento.
- Errores en español: un modelo al que le falta una clave (`ArtifactError`, de las comprobaciones de la carga), el nombre `"mock"` y un nombre que no está en `snake_case`.
- Los archivos de `artifacts/` del proyecto no cambiaron.

In [14]:
import copy
import tempfile
import warnings
from pathlib import Path

from threat_detector.config import ARTIFACTS_DIR, MAX_MODEL_SIZE_MB, MODEL_JOBLIB, MODEL_JSON, MODELS_DIR, ROOT
from threat_detector.artifacts import ArtifactError
from threat_detector.ml.training import BYTES_PER_MB, check_model_size, export_model


def files_snapshot(folder):
    """Ruta, tamaño y fecha de modificación de cada archivo dentro de una carpeta."""
    return {p.relative_to(folder): (p.stat().st_size, p.stat().st_mtime_ns) for p in folder.rglob("*") if p.is_file()}


project_artifacts_before = files_snapshot(ARTIFACTS_DIR)

# Carpeta temporal que hace de artifacts/ para los modelos de prueba; se borra al final del notebook.
temp_artifacts_dir = tempfile.TemporaryDirectory()
temp_artifacts = Path(temp_artifacts_dir.name)
print(f"¿La carpeta temporal está fuera del proyecto?: {not temp_artifacts.is_relative_to(ROOT)}")


def model_info_for(name, display_name, form, detector_type, method):
    """model.json de un modelo de prueba, con su umbral y sus métricas sobre los 2.000 flujos de validación."""
    model, _ = test_models[form]
    threshold = test_infos[form]["threshold"]
    probability, alerts, types = predict_flows(model, test_infos[form], test_flows)
    return build_model_info(**{
        **base_args,
        "name": name, "display_name": display_name, "model": model,
        "detector_type": detector_type, "preprocessing_method": method, "threshold": threshold,
        "threshold_rule": threshold_rule(test_flows[TARGET], probability, threshold),
        "test_metrics": {
            "test_flows": len(test_flows),
            "detection": detection_metrics(test_flows[TARGET], probability, threshold),
            "attack_type": attack_type_metrics(test_flows[ATTACK_CAT], types, alerts),
        },
    })


exports = {
    "A": (test_models["A con 9 clases"][0],
          model_info_for("test_model_a", "Modelo de prueba A (regresión logística y árbol)", "A con 9 clases", "binary", "m2")),
    "B": (test_models["B"][0],
          model_info_for("test_model_b", "Modelo de prueba B (árbol multiclase)", "B", "multiclass", "m3")),
}
written = {}
for form, (model, info) in exports.items():
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        written[form] = export_model(model, info, temp_artifacts)
    model_dir = temp_artifacts / MODELS_DIR.name / info["name"]
    size_mb = (model_dir / MODEL_JOBLIB).stat().st_size / BYTES_PER_MB
    json_text = (model_dir / MODEL_JSON).read_text(encoding="utf-8")
    print(f"\nForma {form}: carpeta {model_dir.relative_to(temp_artifacts)}")
    print(f"  Archivos: {sorted(p.name for p in model_dir.iterdir())}")
    print(f"  model_size_mb = {written[form]['complexity']['model_size_mb']:.4f}; "
          f"¿igual al tamaño en disco ÷ 1.000.000?: {written[form]['complexity']['model_size_mb'] == size_mb}; "
          f"¿menor que {MAX_MODEL_SIZE_MB}?: {size_mb < MAX_MODEL_SIZE_MB}; avisos: {len(caught)}")
    print(f"  ¿model.json leído = lo devuelto?: {json.loads(json_text) == written[form]}; "
          f"¿info recibido sin cambios (model_size_mb = None)?: {info['complexity']['model_size_mb'] is None}")
    print(f"  ¿Tildes legibles en el archivo?: {info['display_name'] in json_text}")

¿La carpeta temporal está fuera del proyecto?: True

Forma A: carpeta models/test_model_a
  Archivos: ['model.joblib', 'model.json']
  model_size_mb = 0.0267; ¿igual al tamaño en disco ÷ 1.000.000?: True; ¿menor que 50?: True; avisos: 0
  ¿model.json leído = lo devuelto?: True; ¿info recibido sin cambios (model_size_mb = None)?: True
  ¿Tildes legibles en el archivo?: True

Forma B: carpeta models/test_model_b
  Archivos: ['model.joblib', 'model.json']
  model_size_mb = 0.0175; ¿igual al tamaño en disco ÷ 1.000.000?: True; ¿menor que 50?: True; avisos: 0
  ¿model.json leído = lo devuelto?: True; ¿info recibido sin cambios (model_size_mb = None)?: True
  ¿Tildes legibles en el archivo?: True


Ahora probamos el aviso de tamaño, la exportación repetida, el caso con `NaN` y los errores. El `NaN` se pone en una copia del `model.json` de la forma A y se exporta con el modelo de la forma A con 10 clases: si `export_model` alcanzara a guardar `model.joblib`, su contenido cambiaría.

In [15]:
# check_model_size con tamaños inventados alrededor del límite.
for size_mb in (MAX_MODEL_SIZE_MB - 0.01, MAX_MODEL_SIZE_MB, MAX_MODEL_SIZE_MB + 0.01):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        check_model_size(size_mb)
    print(f"{size_mb:.2f} MB: " + (f"aviso: {caught[0].message}" if caught else "sin aviso"))

# Exportar de nuevo la forma A con otro umbral reemplaza los archivos.
model_a_dir = temp_artifacts / MODELS_DIR.name / "test_model_a"
model, info = exports["A"]
new_threshold = 0.5
export_model(model, {**info, "threshold": new_threshold}, temp_artifacts)
reloaded = json.loads((model_a_dir / MODEL_JSON).read_text(encoding="utf-8"))
print(f"\nExportada de nuevo: threshold = {reloaded['threshold']} (esperado {new_threshold}); "
      f"archivos: {sorted(p.name for p in model_a_dir.iterdir())}")

# Un NaN en test_metrics, con otro modelo (forma A con 10 clases): si se escribiera, model.joblib cambiaría.
before = {p.name: (p.read_bytes(), p.stat().st_mtime_ns) for p in model_a_dir.iterdir()}
nan_info = copy.deepcopy(reloaded)
nan_info["test_metrics"]["detection"]["pr_auc"] = float("nan")
try:
    export_model(test_models["A con 10 clases"][0], nan_info, temp_artifacts)
    print("NaN en test_metrics: no lanzó error")
except ValueError as error:
    print(f"\nNaN en test_metrics: {error}")
after = {p.name: (p.read_bytes(), p.stat().st_mtime_ns) for p in model_a_dir.iterdir()}
print(f"¿Los dos archivos quedaron idénticos (contenido y fecha de modificación)?: {before == after}")

print()
error_cases = (
    ("Modelo sin attack_type_classifier", {"detector": detector_a}, info),
    ("Nombre mock", model, {**info, "name": "mock"}),
    ("Nombre fuera de snake_case", model, {**info, "name": "Modelo-A"}),
)
for name, bad_model, bad_info in error_cases:
    try:
        export_model(bad_model, bad_info, temp_artifacts)
        print(f"{name}: no lanzó error")
    except (ValueError, ArtifactError) as error:
        print(f"{name}: {error} ({type(error).__name__})")

print(f"\n¿Los archivos de artifacts/ del proyecto no cambiaron?: {files_snapshot(ARTIFACTS_DIR) == project_artifacts_before}")

49.99 MB: sin aviso
50.00 MB: aviso: model.joblib pesa 50.00 MB y el límite es 50 MB. El equipo debe revisarlo antes de subirlo a git.
50.01 MB: aviso: model.joblib pesa 50.01 MB y el límite es 50 MB. El equipo debe revisarlo antes de subirlo a git.

Exportada de nuevo: threshold = 0.5 (esperado 0.5); archivos: ['model.joblib', 'model.json']

NaN en test_metrics: El model.json de test_model_a no se puede escribir como JSON válido: tiene un NaN, un infinito o un valor que no es de Python (por ejemplo, un número de numpy).
¿Los dos archivos quedaron idénticos (contenido y fecha de modificación)?: True

Modelo sin attack_type_classifier: models/test_model_a/model.joblib debe guardar un diccionario con las claves attack_type_classifier y detector. (ArtifactError)
Nombre mock: El nombre mock es del modelo simulado, que se escribe a mano; usa otro nombre. (ValueError)
Nombre fuera de snake_case: El nombre Modelo-A no es válido: debe ir en minúsculas y snake_case (por ejemplo, random_forest).

## 13. Cargar un modelo

La app carga cada modelo registrado una sola vez, al arrancar, con dos funciones de `artifacts.py`:

- `read_model_info`: lee `model.json` y lo comprueba con `check_model_info`. Están todos los campos de la sección 11 del contrato (un campo de más no es error, porque agregar campos no rompe el contrato), `name` es igual al nombre de la carpeta, `is_mock` es booleano, `detector_type` es `"binary"` o `"multiclass"` y `threshold` está entre 0 y 1. En un modelo real no pueden ser `null` los campos que solo lo son en el simulado (`trained_at`, `sklearn_version`, `preprocessing_method`, `threshold_rule` y `complexity`), y con `require_test_metrics=True`, que es lo que pide un modelo registrado, tampoco `test_metrics`. Un `NaN` o un `Infinity` en el archivo se rechaza, porque no son JSON válido.
- `load_model`: carga `model.joblib` y lo comprueba con `check_model`. Revisa las dos claves; que cada `Pipeline` tiene los pasos `"preprocessor"` y `"model"`; que cada preprocesador usa **exactamente** las columnas de `numeric_cols` y `categorical_cols` de `model.json` (así se detecta un `model.json` que no es el de ese `model.joblib`), y que el detector tiene la clase `1` o `"Normal"`, según `detector_type`, y el clasificador los 9 tipos. Con el modelo simulado (`is_mock: true`) devuelve `None` sin buscar `model.joblib`.

Todos los errores son `ArtifactError`, con la ruta relativa del archivo (nunca la del computador) y, cuando vienen de otro error, con la causa original encadenada. `export_model` aplica estas mismas comprobaciones antes de escribir (sección 12).

> **Modelos de prueba en la carpeta temporal.** Cargamos los modelos de prueba que exportamos en la sección 12, que no son modelos del proyecto. Primero los exportamos otra vez, porque la sección 12 cambió el umbral de la forma A.

**Resultado esperado:**
- Formas A y B: `read_model_info` devuelve lo mismo que se escribió, y `predict_flows` con el modelo cargado da exactamente las mismas probabilidades, alertas y tipos que con el modelo original.
- Forma B: después de cargarla, el detector y el clasificador siguen siendo el mismo objeto. Para comprobar que joblib guarda una sola vez un `Pipeline` que aparece en las dos claves, comparamos el tamaño de guardar el mismo `Pipeline` dos veces con el de guardar dos copias: con dos copias el archivo pesa casi el doble.
- Un modelo simulado mínimo (solo `model.json`, con `is_mock: true`): `load_model` devuelve `None`, sin error, aunque no hay `model.joblib`.
- Un modelo exportado con `test_metrics` en `null`: se lee con `require_test_metrics=False` y da error con `True`.

In [16]:
import shutil

import joblib

from threat_detector.artifacts import load_model, read_model_info

loaded_models = {}
for form, (model, info) in exports.items():
    written[form] = export_model(model, info, temp_artifacts)
    name = info["name"]
    read_info = read_model_info(name, temp_artifacts)
    loaded_models[form] = load_model(name, read_info, temp_artifacts)
    original = predict_flows(model, read_info, test_flows)
    reloaded_result = predict_flows(loaded_models[form], read_info, test_flows)
    print(f"Forma {form}: ¿model.json leído = escrito?: {read_info == written[form]}; "
          f"¿mismas probabilidades, alertas y tipos que el original?: "
          f"{all(np.array_equal(a, b) for a, b in zip(original, reloaded_result))}")

loaded_b = loaded_models["B"]
print(f"\nForma B cargada: ¿detector y clasificador son el mismo objeto?: "
      f"{loaded_b['detector'] is loaded_b['attack_type_classifier']}")
shared_path, copies_path = temp_artifacts / "shared.joblib", temp_artifacts / "copies.joblib"
joblib.dump({"detector": multiclass_b, "attack_type_classifier": multiclass_b}, shared_path)
joblib.dump({"detector": multiclass_b, "attack_type_classifier": copy.deepcopy(multiclass_b)}, copies_path)
shared_size, copies_size = shared_path.stat().st_size, copies_path.stat().st_size
print(f"Mismo Pipeline en las dos claves: {shared_size:,} bytes; dos copias: {copies_size:,} bytes "
      f"({copies_size / shared_size:.2f} veces)")
shared_path.unlink()
copies_path.unlink()

# Modelo simulado mínimo: solo model.json, con los campos que en el simulado son null.
mock_dir = temp_artifacts / MODELS_DIR.name / "mock"
mock_dir.mkdir()
mock_info = {**written["A"], "name": "mock", "display_name": "Modelo simulado", "is_mock": True,
             **{field: None for field in ("trained_at", "sklearn_version", "preprocessing_method",
                                          "threshold_rule", "complexity")}}
(mock_dir / MODEL_JSON).write_text(json.dumps(mock_info, ensure_ascii=False), encoding="utf-8")
mock_model = load_model("mock", read_model_info("mock", temp_artifacts), temp_artifacts)
print(f"\nModelo simulado: load_model devuelve {mock_model}; archivos: {sorted(p.name for p in mock_dir.iterdir())}")

# Modelo exportado sin evaluación final (test_metrics = null).
model, info = exports["A"]
export_model(model, {**info, "name": "pending_model", "test_metrics": None}, temp_artifacts)
pending = read_model_info("pending_model", temp_artifacts, require_test_metrics=False)
print(f"\nSin test_metrics, require_test_metrics=False: se lee (test_metrics = {pending['test_metrics']})")
try:
    read_model_info("pending_model", temp_artifacts)
except ArtifactError as error:
    print(f"Sin test_metrics, require_test_metrics=True: {error}")

Forma A: ¿model.json leído = escrito?: True; ¿mismas probabilidades, alertas y tipos que el original?: True
Forma B: ¿model.json leído = escrito?: True; ¿mismas probabilidades, alertas y tipos que el original?: True

Forma B cargada: ¿detector y clasificador son el mismo objeto?: True
Mismo Pipeline en las dos claves: 17,488 bytes; dos copias: 32,533 bytes (1.86 veces)

Modelo simulado: load_model devuelve None; archivos: ['model.json']

Sin test_metrics, require_test_metrics=False: se lee (test_metrics = None)
Sin test_metrics, require_test_metrics=True: models/pending_model/model.json no tiene test_metrics: un modelo registrado debe tener su evaluación final.


## 14. Errores de carga

Si un modelo registrado no se puede cargar, la app no arranca y muestra un error claro con el nombre del modelo y la causa (contrato, sección 10). Aquí provocamos, a propósito, cada problema que debe detectar la carga, sobre copias de los modelos de prueba en la carpeta temporal. Cada copia va en una carpeta con su propio nombre y su `model.json` dice ese nombre, salvo en el caso que prueba un nombre distinto.

Cargar un modelo es `read_model_info` y luego `load_model`, como lo hará la app. Junto a cada error mostramos el tipo de la causa original cuando la hay, para comprobar que se conserva (`raise ArtifactError(...) from error`).

> **Errores provocados a propósito.** Ningún caso toca `artifacts/` del proyecto. El caso de la librería que falta crea un módulo de Python temporal, guarda un objeto suyo en `model.joblib` y borra el módulo antes de cargar, como pasaría si el modelo usara una librería que no está en `requirements-prod.txt`.

**Resultado esperado:** `ArtifactError` en español en cada caso:
- **`model.json`**: falta; está cortado (causa `JSONDecodeError`); trae `NaN` en una métrica (causa `ValueError`); su `name` no es el de la carpeta; le falta un campo; `preprocessing_method` es `null` en un modelo real; `detector_type` desconocido; umbral de 1,5.
- **`model.joblib`**: falta; tiene bytes al azar o está cortado (causa original de joblib o pickle); guarda un diccionario con otras claves; un detector declarado `"binary"` sin la clase `1` (la forma B, que es multiclase; causa `ValueError`); usa una librería que falta (causa `ModuleNotFoundError`).
- **No corresponden**: el `model.json` de un modelo M3 (forma B, declarado `"binary"`) junto al `model.joblib` de uno M2 (forma A): a `numeric_cols` le faltan las 3 variables de TTL que el modelo sí usa.
- **Versión de scikit-learn distinta** (`"1.0.0"` en `model.json`): el modelo **sí** carga, y queda un aviso en el registro con el nombre del modelo y las dos versiones.
- **`export_model`** rechaza la forma B declarada `"binary"` y no crea su carpeta.

In [17]:
import logging
import re
import sys


def copy_model(source, new_name, keep_name=False):
    """Copia la carpeta de un modelo de prueba con otro nombre; su model.json dice el nombre nuevo."""
    target = temp_artifacts / MODELS_DIR.name / new_name
    shutil.copytree(temp_artifacts / MODELS_DIR.name / source, target)
    if not keep_name:
        edit_info(target, name=new_name)
    return target


def edit_info(folder, remove=(), **changes):
    """Cambia campos del model.json de una carpeta (y quita los de remove)."""
    path = folder / MODEL_JSON
    info = json.loads(path.read_text(encoding="utf-8"))
    info.update(changes)
    for field in remove:
        del info[field]
    path.write_text(json.dumps(info, ensure_ascii=False), encoding="utf-8")


def try_load(name):
    """Carga un modelo como lo hará la app: read_model_info y luego load_model."""
    return load_model(name, read_model_info(name, temp_artifacts), temp_artifacts)


broken = []  # (descripción, carpeta)

# model.json
folder = copy_model("test_model_a", "missing_json"); (folder / MODEL_JSON).unlink()
broken.append(("Falta model.json", "missing_json"))
folder = copy_model("test_model_a", "cut_json")
text = (folder / MODEL_JSON).read_text(encoding="utf-8"); (folder / MODEL_JSON).write_text(text[: len(text) // 2], encoding="utf-8")
broken.append(("model.json cortado", "cut_json"))
folder = copy_model("test_model_a", "nan_json")
text = (folder / MODEL_JSON).read_text(encoding="utf-8")
(folder / MODEL_JSON).write_text(re.sub(r'"pr_auc": [^,\n]+', '"pr_auc": NaN', text, count=1), encoding="utf-8")
broken.append(("NaN en una métrica de model.json", "nan_json"))
copy_model("test_model_a", "other_folder", keep_name=True)
broken.append(("name distinto de la carpeta", "other_folder"))
edit_info(copy_model("test_model_a", "missing_field"), remove=["feature_groups"])
broken.append(("Falta un campo", "missing_field"))
edit_info(copy_model("test_model_a", "null_method"), preprocessing_method=None)
broken.append(("preprocessing_method null en un modelo real", "null_method"))
edit_info(copy_model("test_model_a", "bad_detector_type"), detector_type="binario")
broken.append(("detector_type desconocido", "bad_detector_type"))
edit_info(copy_model("test_model_a", "bad_threshold"), threshold=1.5)
broken.append(("Umbral de 1,5", "bad_threshold"))

# model.joblib
folder = copy_model("test_model_a", "missing_joblib"); (folder / MODEL_JOBLIB).unlink()
broken.append(("Falta model.joblib", "missing_joblib"))
folder = copy_model("test_model_a", "random_joblib")
(folder / MODEL_JOBLIB).write_bytes(np.random.default_rng(RANDOM_STATE).bytes(1000))
broken.append(("model.joblib con bytes al azar", "random_joblib"))
folder = copy_model("test_model_a", "cut_joblib")
data = (folder / MODEL_JOBLIB).read_bytes(); (folder / MODEL_JOBLIB).write_bytes(data[: len(data) // 2])
broken.append(("model.joblib cortado", "cut_joblib"))
folder = copy_model("test_model_a", "wrong_keys")
joblib.dump({"detector": detector_a, "classifier": classifier_9}, folder / MODEL_JOBLIB)
broken.append(("model.joblib con otras claves", "wrong_keys"))
edit_info(copy_model("test_model_b", "binary_without_1"), detector_type="binary")
broken.append(("Detector binary sin la clase 1", "binary_without_1"))

# Librería que falta: un módulo temporal que se borra antes de cargar.
folder = copy_model("test_model_a", "missing_library")
module_dir = temp_artifacts / "fake_library_dir"
module_dir.mkdir()
(module_dir / "fake_library.py").write_text("class FakeEstimator:\n    pass\n", encoding="utf-8")
sys.path.insert(0, str(module_dir))
import fake_library
joblib.dump({"detector": fake_library.FakeEstimator(), "attack_type_classifier": fake_library.FakeEstimator()},
            folder / MODEL_JOBLIB)
sys.path.remove(str(module_dir))
del sys.modules["fake_library"]
shutil.rmtree(module_dir)
broken.append(("Librería que falta", "missing_library"))

# model.json de un modelo M3 (forma B, declarado binary) junto al model.joblib de uno M2 (forma A).
folder = copy_model("test_model_a", "json_m3_joblib_m2")
shutil.copy(temp_artifacts / MODELS_DIR.name / "test_model_b" / MODEL_JSON, folder / MODEL_JSON)
edit_info(folder, name="json_m3_joblib_m2", detector_type="binary")
broken.append(("model.json M3 con model.joblib M2", "json_m3_joblib_m2"))

for description, name in broken:
    try:
        try_load(name)
        print(f"{description}: no lanzó error")
    except ArtifactError as error:
        cause = f" [causa original: {type(error.__cause__).__name__}]" if error.__cause__ else ""
        print(f"{description}: {error}{cause}")

Falta model.json: Falta models/missing_json/model.json: el modelo missing_json no tiene su descripción.
model.json cortado: models/cut_json/model.json está dañado: no es un JSON válido. [causa original: JSONDecodeError]
NaN en una métrica de model.json: models/nan_json/model.json está dañado: no es un JSON válido. [causa original: ValueError]
name distinto de la carpeta: models/other_folder/model.json dice name = test_model_a, pero su carpeta se llama other_folder.
Falta un campo: A models/missing_field/model.json le faltan campos: feature_groups.
preprocessing_method null en un modelo real: En models/null_method/model.json son null preprocessing_method, y solo pueden serlo en el modelo simulado.
detector_type desconocido: En models/bad_detector_type/model.json, detector_type = binario; debe ser uno de: binary, multiclass.
Umbral de 1,5: En models/bad_threshold/model.json, threshold = 1.5; debe ser un número entre 0 y 1.
Falta model.joblib: Falta models/missing_joblib/model.joblib: el 

Por último, la versión de scikit-learn: si no coincide con la instalada, la app solo avisa en el registro del servidor y el modelo se carga. Para mostrar el aviso aquí, capturamos el registro de `artifacts.py` mientras se carga. También comprobamos que `export_model` usa las mismas comprobaciones que la carga y no escribe nada si fallan.

In [18]:
class ListHandler(logging.Handler):
    """Guarda los mensajes del registro para poder mostrarlos en el notebook."""

    def __init__(self):
        super().__init__()
        self.messages = []

    def emit(self, record):
        self.messages.append(f"{record.levelname}: {record.getMessage()}")


# Versión de scikit-learn distinta: el modelo carga y queda un aviso en el registro.
edit_info(copy_model("test_model_a", "old_sklearn"), sklearn_version="1.0.0")
artifacts_logger = logging.getLogger("threat_detector.artifacts")
handler = ListHandler()
artifacts_logger.addHandler(handler)
artifacts_logger.propagate = False  # solo se muestra aquí, una vez
try:
    old_model = try_load("old_sklearn")
finally:
    artifacts_logger.removeHandler(handler)
    artifacts_logger.propagate = True
print(f"Versión distinta: ¿cargó?: {old_model is not None}; registro: {handler.messages}")

# export_model rechaza la forma B declarada binary, sin crear su carpeta.
model, info = exports["B"]
try:
    export_model(model, {**info, "name": "test_model_b_binary", "detector_type": "binary"}, temp_artifacts)
    print("export_model con la forma B declarada binary: no lanzó error")
except ArtifactError as error:
    print(f"\nexport_model con la forma B declarada binary: {error}")
print(f"¿Se creó su carpeta?: {(temp_artifacts / MODELS_DIR.name / 'test_model_b_binary').exists()}")

Versión distinta: ¿cargó?: True; registro: ['WARNING: El modelo old_sklearn se entrenó con scikit-learn 1.0.0 y está instalada la versión 1.9.1; puede comportarse distinto.']

export_model con la forma B declarada binary: En models/test_model_b_binary/model.joblib, detector no tiene las clases que pide el contrato (detector_type = binary): El modelo no tiene la clase 1 entre sus clases: Analysis, Backdoor, DoS, Exploits, Fuzzers, Generic, Normal, Reconnaissance, Shellcode, Worms.
¿Se creó su carpeta?: False


## 15. Registro

`registry.json` dice qué modelos puede usar la app, en qué orden, y cuál es el modelo por defecto (contrato, sección 12). Las funciones son:

- `read_registry` (en `artifacts.py`): lee y comprueba el registro. Debe existir y ser JSON válido, con `default_model` y `models`; cada modelo de `models` es un objeto con `name`, sin nombres repetidos, y `default_model` es uno de ellos.
- `load_artifacts` (en `artifacts.py`): lo único que llamará la app al arrancar. Lee el registro y carga cada modelo registrado con `read_model_info` (exigiendo `test_metrics`) y `load_model`, en el orden del registro. Las carpetas que no están registradas se ignoran. Si algo falla, lanza `ArtifactError` y la app no debe arrancar.
- `register_model` (en `ml/training.py`): agrega un modelo después de su evaluación final. Antes comprueba que la app lo podría cargar, con `test_metrics`. Si ya está registrado, no hace nada; si no, lo agrega antes del modelo simulado (si está) o al final, y no cambia `default_model`. Escribe primero un archivo temporal en la misma carpeta y luego lo cambia por `registry.json` con `os.replace`, para que el registro nunca quede escrito a medias. `registry.json` debe existir: es un archivo del proyecto.
- `export_model`: si el modelo ya está registrado, exige `test_metrics`, porque sin ellas la app no podría arrancar. Si no hay `registry.json`, ningún modelo está registrado: así exportamos en las secciones 12 a 14, cuando la carpeta temporal aún no tenía registro (incluido `pending_model`, sin `test_metrics`).

> **Registro de prueba en la carpeta temporal.** Escribimos a mano un `registry.json` en la carpeta temporal, con solo el modelo simulado mínimo de la sección 13 como modelo por defecto, y registramos ahí los modelos de prueba. El `registry.json` del proyecto no se toca.

**Resultado esperado:**
- Registro inicial: `load_artifacts` devuelve solo `mock`, con modelo `None`.
- Registrar la forma A y luego la B: el orden queda `test_model_a`, `test_model_b`, `mock`, y `default_model` sigue siendo `mock`. Registrar A otra vez deja el archivo idéntico (contenido y fecha de modificación).
- `load_artifacts` devuelve los tres modelos en ese orden, e ignora las carpetas sin registrar (las de la sección 14 y `pending_model`).
- Sin `mock` en el registro, el modelo nuevo va al final.
- `register_model` rechaza, sin tocar el archivo: un modelo sin `test_metrics` (`pending_model`), uno dañado (`random_joblib`) y uno que no existe.
- Si escribir el archivo temporal falla (aquí, porque ya existe una carpeta con su nombre), `registry.json` queda idéntico.
- Exportar la forma A, ya registrada, con `test_metrics` en `null`: error, y sus dos archivos quedan idénticos. Con métricas se exporta. Con un `registry.json` dañado, `export_model` también da error.
- Errores de `read_registry`: falta el archivo, no es JSON válido, falta `default_model`, modelos repetidos, `default_model` fuera de `models` y un modelo sin `name`.
- Errores de `load_artifacts`: un modelo registrado sin carpeta y un modelo registrado sin `test_metrics` (con el registro editado a mano).

In [19]:
from threat_detector.artifacts import load_artifacts, read_registry
from threat_detector.config import REGISTRY_PATH
from threat_detector.ml.training import TEMP_SUFFIX, register_model

registry_path = temp_artifacts / REGISTRY_PATH.name


def write_registry(registry):
    """Escribe a mano el registry.json de la carpeta temporal."""
    registry_path.write_text(json.dumps(registry, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


def registry_state():
    """Contenido y fecha de modificación de registry.json, para saber si cambió."""
    return registry_path.read_bytes(), registry_path.stat().st_mtime_ns


write_registry({"default_model": "mock", "models": [{"name": "mock"}]})
registry, models = load_artifacts(temp_artifacts)
print(f"Registro inicial: modelos {list(models)}; ¿modelo del simulado es None?: {models['mock']['model'] is None}")

register_model("test_model_a", temp_artifacts)
registry = register_model("test_model_b", temp_artifacts)
print(f"\nDespués de registrar A y B: {[m['name'] for m in registry['models']]}; default_model = {registry['default_model']}")
before = registry_state()
register_model("test_model_a", temp_artifacts)
print(f"Registrar A otra vez: ¿archivo idéntico (contenido y fecha)?: {registry_state() == before}")
print(f"¿El archivo leído es el devuelto?: {read_registry(temp_artifacts) == registry}")

registry, models = load_artifacts(temp_artifacts)
folders = sorted(p.name for p in (temp_artifacts / MODELS_DIR.name).iterdir())
ignored = [f for f in folders if f not in models]
print(f"\nload_artifacts: {list(models)}; carpetas sin registrar ignoradas: {len(ignored)} "
      f"(entre ellas pending_model: {'pending_model' in ignored})")
same = all(np.array_equal(a, b) for a, b in zip(
    predict_flows(models["test_model_b"]["model"], models["test_model_b"]["info"], test_flows),
    predict_flows(exports["B"][0], written["B"], test_flows)))
print(f"¿La forma B cargada por load_artifacts predice igual que la original?: {same}")

# Sin mock en el registro, el modelo nuevo va al final.
full_registry = registry_path.read_bytes()
write_registry({"default_model": "test_model_b", "models": [{"name": "test_model_b"}]})
print(f"\nSin mock: {[m['name'] for m in register_model('test_model_a', temp_artifacts)['models']]}")
registry_path.write_bytes(full_registry)

print()
before = registry_state()
for name in ("pending_model", "random_joblib", "ghost_model"):
    try:
        register_model(name, temp_artifacts)
        print(f"register_model({name}): no lanzó error")
    except ArtifactError as error:
        print(f"register_model({name}): {error}")
print(f"¿registry.json quedó idéntico?: {registry_state() == before}")

# Falla al escribir el archivo temporal: una carpeta con su nombre impide escribirlo.
model, info = exports["A"]
export_model(model, {**info, "name": "test_model_c", "display_name": "Modelo de prueba C"}, temp_artifacts)
blocker = registry_path.with_name(registry_path.name + TEMP_SUFFIX)
blocker.mkdir()
before = registry_state()
try:
    register_model("test_model_c", temp_artifacts)
    print("\nFalla al escribir el temporal: no lanzó error")
except OSError as error:
    print(f"\nFalla al escribir el temporal: {type(error).__name__}; ¿registry.json quedó idéntico?: {registry_state() == before}")
blocker.rmdir()

Registro inicial: modelos ['mock']; ¿modelo del simulado es None?: True

Después de registrar A y B: ['test_model_a', 'test_model_b', 'mock']; default_model = mock
Registrar A otra vez: ¿archivo idéntico (contenido y fecha)?: True
¿El archivo leído es el devuelto?: True

load_artifacts: ['test_model_a', 'test_model_b', 'mock']; carpetas sin registrar ignoradas: 17 (entre ellas pending_model: True)
¿La forma B cargada por load_artifacts predice igual que la original?: True

Sin mock: ['test_model_b', 'test_model_a']

register_model(pending_model): models/pending_model/model.json no tiene test_metrics: un modelo registrado debe tener su evaluación final.
register_model(random_joblib): No se pudo cargar models/random_joblib/model.joblib: el archivo está dañado.
register_model(ghost_model): Falta models/ghost_model/model.json: el modelo ghost_model no tiene su descripción.
¿registry.json quedó idéntico?: True

Falla al escribir el temporal: IsADirectoryError; ¿registry.json quedó idéntico?

Ahora comprobamos que un modelo ya registrado no se puede exportar sin `test_metrics`. Después, los errores del propio `registry.json` y de `load_artifacts`: cada caso escribe a mano un registro con el problema, lo prueba y deja el registro como estaba.

In [20]:
model, info = exports["A"]
model_a_dir = temp_artifacts / MODELS_DIR.name / "test_model_a"
before = {p.name: (p.read_bytes(), p.stat().st_mtime_ns) for p in model_a_dir.iterdir()}
try:
    export_model(model, {**info, "test_metrics": None}, temp_artifacts)
    print("Exportar A (registrada) sin test_metrics: no lanzó error")
except ArtifactError as error:
    print(f"Exportar A (registrada) sin test_metrics: {error}")
after = {p.name: (p.read_bytes(), p.stat().st_mtime_ns) for p in model_a_dir.iterdir()}
print(f"¿Sus dos archivos quedaron idénticos?: {before == after}")
print(f"Exportar A (registrada) con test_metrics: ¿se exportó?: {export_model(model, info, temp_artifacts)['name'] == 'test_model_a'}")

good_registry = registry_path.read_bytes()
registry_cases = (
    ("Falta registry.json", None),
    ("registry.json no es JSON válido", '{"default_model": "mock", "models": ['),
    ("Falta default_model", {"models": [{"name": "mock"}]}),
    ("Modelos repetidos", {"default_model": "mock", "models": [{"name": "mock"}, {"name": "mock"}]}),
    ("default_model fuera de models", {"default_model": "test_model_a", "models": [{"name": "mock"}]}),
    ("Un modelo sin name", {"default_model": "mock", "models": [{"name": "mock"}, {"model": "test_model_a"}]}),
)
print()
for description, content in registry_cases:
    if content is None:
        registry_path.unlink()
    elif isinstance(content, str):
        registry_path.write_text(content, encoding="utf-8")
    else:
        write_registry(content)
    try:
        read_registry(temp_artifacts)
        print(f"{description}: no lanzó error")
    except ArtifactError as error:
        print(f"{description}: {error}")
    registry_path.write_bytes(good_registry)

registry_path.write_text("{", encoding="utf-8")
try:
    export_model(model, info, temp_artifacts)
    print("export_model con registry.json dañado: no lanzó error")
except ArtifactError as error:
    print(f"export_model con registry.json dañado: {error}")
registry_path.write_bytes(good_registry)

print()
for description, extra in (("Registrado sin carpeta", "ghost_model"), ("Registrado sin test_metrics", "pending_model")):
    write_registry({"default_model": "mock", "models": [{"name": "mock"}, {"name": extra}]})
    try:
        load_artifacts(temp_artifacts)
        print(f"load_artifacts, {description.lower()}: no lanzó error")
    except ArtifactError as error:
        print(f"load_artifacts, {description.lower()}: {error}")
    registry_path.write_bytes(good_registry)

print(f"\n¿El registro de prueba quedó como estaba?: {[m['name'] for m in read_registry(temp_artifacts)['models']]}")
print(f"¿Los archivos de artifacts/ del proyecto no cambiaron?: {files_snapshot(ARTIFACTS_DIR) == project_artifacts_before}")

Exportar A (registrada) sin test_metrics: models/test_model_a/model.json no tiene test_metrics: un modelo registrado debe tener su evaluación final.
¿Sus dos archivos quedaron idénticos?: True
Exportar A (registrada) con test_metrics: ¿se exportó?: True

Falta registry.json: Falta registry.json: la app no sabe qué modelos puede usar.
registry.json no es JSON válido: registry.json está dañado: no es un JSON válido.
Falta default_model: A registry.json le faltan campos: default_model.
Modelos repetidos: En registry.json hay modelos repetidos: mock.
default_model fuera de models: En registry.json, default_model = test_model_a no es uno de los modelos de models.
Un modelo sin name: En registry.json, models debe ser una lista de objetos, cada uno con su name.
export_model con registry.json dañado: registry.json está dañado: no es un JSON válido.

load_artifacts, registrado sin carpeta: Falta models/ghost_model/model.json: el modelo ghost_model no tiene su descripción.
load_artifacts, regist

## 16. Modelo simulado y registro del proyecto

Mientras no haya modelos reales, `artifacts/` del proyecto tiene solo el modelo simulado (contrato, sección 13): `artifacts/models/mock/model.json`, **sin** `model.joblib`, y `artifacts/registry.json` con el simulado como único modelo y modelo por defecto. Los dos archivos se escribieron a mano, copiando los ejemplos del contrato; ningún código del proyecto los genera. Esta sección solo los lee.

Comprobamos que coinciden con el contrato y con el código:

1. **Carga:** `load_artifacts()` con el `artifacts/` del proyecto carga sin errores y devuelve solo `mock`, sin modelo (la app usará `mock.py`).
2. **`registry.json`**: igual al primer ejemplo de la sección 12 del contrato.
3. **`model.json` frente al contrato:** los campos de la sección 11, en orden, con los valores de la tabla de la sección 13; `test_metrics` idéntico al ejemplo de la sección 7, y los campos del ejemplo de `GET /api/models` (sección 6) iguales a los del simulado.
4. **`model.json` frente al código:** `numeric_cols` = `numeric_input_cols(use_ttl=True)`, `categorical_cols` = `CATEGORICAL_COLS`, `log_cols` = las de `LOG_COLS` en el orden del archivo y `feature_groups` = el del preprocesador M2 ajustado con toda la parte de entrenamiento (sección 11). `test_metrics` tiene la misma estructura que la salida de `detection_metrics` y `attack_type_metrics`.
5. **Métricas posibles para la prueba real:** aunque son inventadas, deben poder salir de la prueba. Con `load_test` contamos los flujos reales: `test_flows` = `TEST_ROWS`; `tp + fn` = ataques reales; `fp + tn` = normales reales, y el `support` de cada tipo es menor o igual que sus flujos reales (los que faltan son ataques sin alerta, y suman `fn`). La prueba solo se usa para contar etiquetas; no sale de aquí ningún valor del proyecto.
6. **Métricas coherentes entre sí:** recalculamos desde la matriz de confusión entre tipos y desde `confusion` todas las métricas (`precision`, `recall`, `f1` y `support` de cada tipo, `macro_f1`, `evaluated_flows`, `detection_rate`, `false_alarm_rate`, `precision`, `f1` y `accuracy`) y las comparamos con las del archivo, redondeadas a 4 decimales como en el contrato. `pr_auc` y `roc_auc` no se pueden recalcular, porque necesitan la probabilidad de cada flujo.

Al final borramos la carpeta temporal de los modelos de prueba (secciones 12 a 15) y comprobamos que `artifacts/` del proyecto no cambió durante el notebook.

**Resultado esperado:** todas las comprobaciones dan `True`, y la carpeta temporal ya no existe.

In [21]:
from threat_detector.artifacts import MODEL_INFO_FIELDS
from threat_detector.config import TEST_ROWS
from threat_detector.ml.preprocessing import LOG_COLS, numeric_input_cols

# 1. Carga del artifacts/ del proyecto, como lo hará la app.
project_registry, project_models = load_artifacts()
mock = project_models["mock"]["info"]
print(f"1. load_artifacts(): modelos {list(project_models)}; ¿el simulado no tiene modelo?: "
      f"{project_models['mock']['model'] is None}; ¿sin model.joblib en su carpeta?: "
      f"{not (MODELS_DIR / 'mock' / MODEL_JOBLIB).exists()}")

# 2. registry.json frente al primer ejemplo de la sección 12.
registry_example = [example for section, example in parsed if section.startswith("12.")][0]
print(f"2. ¿registry.json igual al ejemplo de la sección 12?: {project_registry == registry_example}")

# 3. model.json frente al contrato.
table_13 = {
    "name": "mock", "display_name": "Modelo simulado", "is_mock": True, "trained_at": None,
    "sklearn_version": None, "preprocessing_method": None, "threshold_rule": None, "complexity": None,
    "detector_type": "binary", "use_ttl": True, "threshold": 0.31, "can_explain": True,
    "explanation_method": "mock",
}
metrics_example = [example for section, example in parsed if section.startswith("7.")][0]["models"][0]
models_example = [example for section, example in parsed if section.startswith("6.")][0]["models"][0]
print(f"3. ¿Campos de la sección 11, en orden?: {list(mock) == MODEL_INFO_FIELDS}; "
      f"¿valores de la tabla de la sección 13?: {all(mock[k] == v for k, v in table_13.items())}")
print(f"   ¿test_metrics idéntico al ejemplo de la sección 7?: "
      f"{mock['test_metrics'] == {k: metrics_example[k] for k in ('test_flows', 'detection', 'attack_type')}}; "
      f"¿coincide con el ejemplo de GET /api/models?: {all(mock[k] == v for k, v in models_example.items())}")
print(f"   ¿Listas de variables iguales al ejemplo de la sección 11?: "
      f"{all(mock[k] == model_json[k] for k in ('numeric_cols', 'categorical_cols', 'log_cols', 'feature_groups'))}")

# 4. model.json frente al código.
numeric_ttl = numeric_input_cols(use_ttl=True)
print(f"4. ¿numeric_cols = numeric_input_cols(True)?: {mock['numeric_cols'] == numeric_ttl}; "
      f"¿categorical_cols = CATEGORICAL_COLS?: {mock['categorical_cols'] == CATEGORICAL_COLS}; "
      f"¿log_cols = LOG_COLS en el orden del archivo?: {mock['log_cols'] == [c for c in numeric_ttl if c in LOG_COLS]}")
print(f"   ¿feature_groups = el del preprocesador M2 de la sección 11 (contenido y orden)?: "
      f"{list(mock['feature_groups'].items()) == list(groups_m2.items())}; "
      f"¿test_metrics con la estructura de las funciones de métricas?: "
      f"{structure(mock['test_metrics']) == structure(base_args['test_metrics'])}")

1. load_artifacts(): modelos ['mock']; ¿el simulado no tiene modelo?: True; ¿sin model.joblib en su carpeta?: True
2. ¿registry.json igual al ejemplo de la sección 12?: True
3. ¿Campos de la sección 11, en orden?: True; ¿valores de la tabla de la sección 13?: True
   ¿test_metrics idéntico al ejemplo de la sección 7?: True; ¿coincide con el ejemplo de GET /api/models?: True
   ¿Listas de variables iguales al ejemplo de la sección 11?: True
4. ¿numeric_cols = numeric_input_cols(True)?: True; ¿categorical_cols = CATEGORICAL_COLS?: True; ¿log_cols = LOG_COLS en el orden del archivo?: True
   ¿feature_groups = el del preprocesador M2 de la sección 11 (contenido y orden)?: True; ¿test_metrics con la estructura de las funciones de métricas?: True


Ahora las métricas inventadas del simulado: primero, que podrían salir de la prueba real (solo contamos sus etiquetas); después, que son coherentes entre sí, recalculándolas desde la matriz y desde `confusion`.

In [22]:
# 5. Métricas posibles para la prueba real: solo se cuentan sus etiquetas.
test_counts = load_test()[ATTACK_CAT].value_counts()
real_attacks = int(test_counts.drop(NORMAL_CAT).sum())
real_normals = int(test_counts[NORMAL_CAT])
detection = mock["test_metrics"]["detection"]
attack_type = mock["test_metrics"]["attack_type"]
confusion = detection["confusion"]
print(f"5. test_flows = {mock['test_metrics']['test_flows']:,} (TEST_ROWS = {TEST_ROWS:,})")
print(f"   tp + fn = {confusion['tp'] + confusion['fn']:,} (ataques reales: {real_attacks:,}); "
      f"fp + tn = {confusion['fp'] + confusion['tn']:,} (normales reales: {real_normals:,})")
supports = pd.DataFrame(
    {"support": {c["attack_cat"]: c["support"] for c in attack_type["per_class"]}, "reales": test_counts[ATTACK_TYPES]}
)
supports["sin alerta"] = supports["reales"] - supports["support"]
print(f"   ¿support ≤ flujos reales en los 9 tipos?: {bool((supports['sin alerta'] >= 0).all())}; "
      f"¿los sin alerta suman fn?: {supports['sin alerta'].sum() == confusion['fn']}")

# 6. Métricas recalculadas desde la matriz y desde confusion, redondeadas a 4 decimales.
DECIMALS = 4
matrix = np.array(attack_type["confusion"]["matrix"])
rows_ok, f1_values = [], []
for i, entry in enumerate(attack_type["per_class"]):
    tp_i, support_i, predicted_i = matrix[i, i], matrix[i].sum(), matrix[:, i].sum()
    f1_values.append(2 * tp_i / (support_i + predicted_i))
    recalculated = {"attack_cat": ATTACK_TYPES[i], "precision": round(tp_i / predicted_i, DECIMALS),
                    "recall": round(tp_i / support_i, DECIMALS), "f1": round(f1_values[-1], DECIMALS),
                    "support": int(support_i)}
    rows_ok.append(recalculated == entry)
tp, fp, tn, fn = (confusion[k] for k in ("tp", "fp", "tn", "fn"))
recalculated_detection = {
    "detection_rate": round(tp / (tp + fn), DECIMALS), "false_alarm_rate": round(fp / (fp + tn), DECIMALS),
    "precision": round(tp / (tp + fp), DECIMALS), "f1": round(2 * tp / (2 * tp + fp + fn), DECIMALS),
    "accuracy": round((tp + tn) / (tp + fp + tn + fn), DECIMALS),
}
print(f"\n6. ¿labels = ATTACK_TYPES?: {attack_type['confusion']['labels'] == ATTACK_TYPES}; "
      f"¿per_class de los 9 tipos igual al recalculado?: {all(rows_ok)}")
print(f"   ¿macro_f1 igual al recalculado?: {round(float(np.mean(f1_values)), DECIMALS) == attack_type['macro_f1']}; "
      f"¿evaluated_flows = suma de la matriz = tp?: {attack_type['evaluated_flows'] == matrix.sum() == tp}")
print(f"   ¿Métricas de detección iguales a las recalculadas?: "
      f"{all(detection[k] == v for k, v in recalculated_detection.items())}")

# Limpieza: se borra la carpeta temporal y se comprueba que artifacts/ del proyecto no cambió.
temp_artifacts_dir.cleanup()
print(f"\n¿La carpeta temporal ya no existe?: {not temp_artifacts.exists()}; "
      f"¿artifacts/ del proyecto sin cambios durante el notebook?: {files_snapshot(ARTIFACTS_DIR) == project_artifacts_before}")
supports

5. test_flows = 82,332 (TEST_ROWS = 82,332)
   tp + fn = 45,332 (ataques reales: 45,332); fp + tn = 37,000 (normales reales: 37,000)
   ¿support ≤ flujos reales en los 9 tipos?: True; ¿los sin alerta suman fn?: True

6. ¿labels = ATTACK_TYPES?: True; ¿per_class de los 9 tipos igual al recalculado?: True
   ¿macro_f1 igual al recalculado?: True; ¿evaluated_flows = suma de la matriz = tp?: True
   ¿Métricas de detección iguales a las recalculadas?: True

¿La carpeta temporal ya no existe?: True; ¿artifacts/ del proyecto sin cambios durante el notebook?: True


,support,reales,sin alerta
Generic,18850,18871,21
Exploits,10575,11132,557
Fuzzers,4850,6062,1212
DoS,3925,4089,164
Reconnaissance,3251,3496,245
Analysis,609,677,68
Backdoor,531,583,52
Shellcode,355,378,23
Worms,41,44,3
